# Phase 2 imbalance sweep - notebook 17 of 20

Runs shards **85-89** (50 replications, indices
`850`-`899`) of both Phase 2 experiments:

* **Part A** - false positives under covariate shift: the causal nulls hold
  exactly while `L(D|A=1) != L(D|A=0)`, swept over six propensity strengths.
* **Part B** - Simpson masking: `L(D|A=1) = L(D|A=0)` exactly while
  `psi_d != 0`.

**Runtime -> Run all**, then leave the tab open. Expect roughly 1.5-2 h. Each
shard downloads a `phase2_shard<i>.json` the moment it finishes, so a dropped
session costs at most the shard in flight. Drop every downloaded file into
`results/shards/` in the repo and run

```
python experiments/phase2_imbalance_sweep.py --mode aggregate
```

once the whole fleet is in.


In [ ]:
%%bash
pip install -q numpy scipy scikit-learn joblib matplotlib scikit-fda \
              gudhi ripser persim pot
pip install -q git+https://github.com/hugogobato/tcda_uq.git
echo '--- install done ---'


In [ ]:
import os

for d in ['experiments', 'tda2s', 'tda2s/adapters', 'tda2s/benchmarks', 'tda2s/dgp', 'tda2s/ph', 'tda2s/resample', 'tda2s/vec', 'results/shards']:
    os.makedirs(d, exist_ok=True)
print("package tree ready")


In [ ]:
%%writefile tda2s/__init__.py
"""tda2s: topological two-sample testing infrastructure (shared by P1 and P2)."""

__version__ = "0.0.1"

In [ ]:
%%writefile tda2s/dgp/__init__.py
"""Controlled DGP harness: point-cloud generators and two-group datasets.

Public API:
  * generators: ``circle_cloud``, ``torus_cloud``, ``sphere_cloud``,
    ``cluster_cloud``, ``loops_cloud``, ``split_cluster_cloud`` (the WP1.1 /
    Phase 4.4 cluster-splitting witness);
  * harness: ``CloudSampleDGP`` (two-group datasets with covariate-driven
    topology, propensity, and group-effect knobs), ``CloudSample`` (the
    realised dataset with per-cloud oracles), ``masking_stratum_sample``
    (the Phase 2.2 exact Simpson-masking DGP: H0^cond true, H0^out false);
  * export: ``to_silhouette_sample`` (clouds -> tcda_uq-format ``(phi, A, X)``).
"""

from .clouds import (circle_cloud, cluster_cloud, loops_cloud, sphere_cloud,
                     split_cluster_cloud, torus_cloud)
from .simulation import (CloudSample, CloudSampleDGP, masking_stratum_sample,
                         to_silhouette_sample)

__all__ = [
    "circle_cloud",
    "torus_cloud",
    "sphere_cloud",
    "cluster_cloud",
    "loops_cloud",
    "split_cluster_cloud",
    "CloudSampleDGP",
    "CloudSample",
    "masking_stratum_sample",
    "to_silhouette_sample",
]

In [ ]:
%%writefile tda2s/dgp/clouds.py
"""Controlled point-cloud generators (Phase 0.6).

Basic shape generators (circle, torus, sphere, cluster) plus the loop cloud
used by the covariate-driven DGP harness: ``n_loops`` circles arranged on a big
circle, each with its own radius, dialable noise and outlier fraction.

All generators return ``(n, d)`` float arrays (2-D for circle / cluster /
loops, 3-D for torus / sphere) and accept an ``rng`` argument that may be an
integer seed, ``None``, or a ``numpy`` Generator (a Generator is used as-is so
streams stay composable).

Memory discipline: generators are vectorised and allocate only ``O(n)``; they
are intended for small clouds (``n <= 300``).
"""

from __future__ import annotations

import numpy as np


def _as_rng(rng):
    """Normalise ``rng`` to a ``numpy`` Generator (Generator passthrough)."""
    return rng if isinstance(rng, np.random.Generator) else np.random.default_rng(rng)


def circle_cloud(n, radius=1.0, noise=0.05, rng=None):
    """Noisy circle: ``n`` points on a circle of radius ``radius`` plus Gaussian jitter.

    Args:
        n: number of points.
        radius: circle radius (the persistent ``H_1`` feature scale).
        noise: standard deviation of the isotropic Gaussian jitter.
        rng: seed or Generator.

    Returns:
        ``(n, 2)`` point cloud with one prominent ``H_1`` feature of persistence
        approximately ``radius``.
    """
    rng = _as_rng(rng)
    theta = rng.uniform(0.0, 2.0 * np.pi, size=n)
    pts = np.column_stack([radius * np.cos(theta), radius * np.sin(theta)])
    return pts + rng.normal(scale=noise, size=pts.shape)


def torus_cloud(n, R=2.0, r=0.6, noise=0.05, rng=None):
    """Noisy torus: ``n`` points on a 3-D torus with major radius ``R`` and minor radius ``r``.

    Parametrised by ``u, v ~ Uniform[0, 2 pi)``:
    ``x = (R + r cos v) cos u``, ``y = (R + r cos v) sin u``, ``z = r sin v``.

    Returns:
        ``(n, 3)`` point cloud with one prominent ``H_2`` feature and one
        prominent ``H_1`` feature (the two independent cycles of the torus).
    """
    rng = _as_rng(rng)
    u = rng.uniform(0.0, 2.0 * np.pi, size=n)
    v = rng.uniform(0.0, 2.0 * np.pi, size=n)
    x = (R + r * np.cos(v)) * np.cos(u)
    y = (R + r * np.cos(v)) * np.sin(u)
    z = r * np.sin(v)
    pts = np.column_stack([x, y, z])
    return pts + rng.normal(scale=noise, size=pts.shape)


def sphere_cloud(n, radius=1.0, noise=0.05, rng=None):
    """Noisy 2-sphere: ``n`` points uniform on the sphere surface plus Gaussian jitter.

    Uses the standard ``z ~ Uniform(-1, 1)``, ``phi ~ Uniform(0, 2 pi)``
    parametrisation.

    Returns:
        ``(n, 3)`` point cloud with one prominent ``H_2`` feature of scale
        approximately ``radius``.
    """
    rng = _as_rng(rng)
    z = rng.uniform(-1.0, 1.0, size=n)
    phi = rng.uniform(0.0, 2.0 * np.pi, size=n)
    rho = np.sqrt(np.maximum(0.0, 1.0 - z**2))
    pts = radius * np.column_stack([rho * np.cos(phi), rho * np.sin(phi), z])
    return pts + rng.normal(scale=noise, size=pts.shape)


def cluster_cloud(n, n_clusters=3, spread=3.0, noise=0.2, rng=None):
    """Gaussian blob clusters: ``n`` points split across ``n_clusters`` blobs.

    Cluster centers sit on a lattice with spacing ``spread``; points within a
    cluster are Gaussian with standard deviation ``noise``.

    Returns:
        ``(n, 2)`` point cloud with ``n_clusters`` connected components at the
        ``H_0`` scale ``spread``.
    """
    rng = _as_rng(rng)
    if n_clusters < 1:
        raise ValueError("n_clusters must be >= 1")
    side = int(np.ceil(np.sqrt(n_clusters)))
    centers = []
    for k in range(n_clusters):
        i, j = k % side, k // side
        centers.append([(i - (side - 1) / 2) * spread, (j - (side - 1) / 2) * spread])
    centers = np.asarray(centers, dtype=float)
    sizes = np.full(n_clusters, n // n_clusters)
    sizes[: n % n_clusters] += 1
    pts = []
    for k, size in enumerate(sizes):
        pts.append(centers[k] + rng.normal(scale=noise, size=(size, 2)))
    return np.vstack(pts)


def split_cluster_cloud(n_per_blob, n_blobs, separation=3.0, noise=0.15,
                        deterministic=False, n_gon=12, rng=None):
    """Cloud of ``n_blobs`` Gaussian blobs on a regular polygon of side ``separation``.

    This is the WP1.1 / Phase 4.4 "cluster splitting" DGP: the H_0 diagram of a
    cloud with ``n_blobs`` well-separated blobs has ``n_blobs - 1`` finite
    classes, all dying at the blob-merge scale. Two blobs (one finite class)
    versus three blobs (two finite classes, equilateral arrangement so both
    classes share the merge law) give equal mean power-weighted silhouettes but
    different diagram laws; the mean is preserved because the silhouette is a
    normalized average and the merge-scale law is unchanged by the extra blob.

    With ``deterministic=True`` each blob is a fixed regular ``n_gon``-gon of
    radius ``noise`` (degenerate randomness), so under the radius-convention
    filtrations of ``tda2s.ph`` (alpha, Delaunay-Cech) the merge scale is
    exactly ``(separation - 2 * noise) / 2`` and the mean-silhouette equality
    holds realization by realization, not only in expectation. Requires
    ``n_gon`` divisible by 4 for ``n_blobs`` in {2, 3}, so that a vertex sits
    exactly on the inter-blob axis in both arrangements; with the default
    ``n_gon=12`` the within-blob classes all die at
    ``noise * sin(pi / n_gon)`` (0.0388 at the defaults), well below the
    persistence threshold used to isolate the merge classes.

    Note that the two arms differ in cardinality (``n_blobs * n_gon`` points),
    so any use of this generator as a null DGP must either state cloud size as
    part of the treatment or equalise it by subsampling.

    Args:
        n_per_blob: points per blob in the stochastic case (Gaussian around the
            blob centre with scale ``noise``). Ignored when
            ``deterministic=True``, which always emits ``n_gon`` vertices.
        n_blobs: number of blobs, placed at the vertices of a regular polygon
            with side length ``separation`` (2 blobs: a segment; 3 blobs: an
            equilateral triangle; more: the regular polygon, so consecutive
            neighbours are at distance ``separation``).
        separation: distance between neighbouring blob centres.
        noise: per-blob scale: Gaussian standard deviation (stochastic) or
            blob radius (deterministic).
        deterministic: use fixed regular polygons instead of Gaussian blobs.
        n_gon: vertices per blob in the deterministic case (default 12).
        rng: seed or Generator.

    Returns:
        ``(n, 2)`` point cloud.
    """
    rng = _as_rng(rng)
    n_blobs = int(n_blobs)
    if n_blobs < 2:
        raise ValueError("n_blobs must be >= 2 (one blob has no finite H_0 class)")
    # Vertices of a regular n-gon with consecutive side length `separation`.
    R = separation / (2.0 * np.sin(np.pi / n_blobs))
    angles = 2.0 * np.pi * np.arange(n_blobs) / n_blobs
    centers = R * np.column_stack([np.cos(angles), np.sin(angles)])

    pts = []
    for k in range(n_blobs):
        if deterministic:
            theta = 2.0 * np.pi * np.arange(int(n_gon)) / int(n_gon)
            blob = centers[k] + noise * np.column_stack([np.cos(theta), np.sin(theta)])
        else:
            blob = centers[k] + rng.normal(scale=noise, size=(n_per_blob, 2))
        pts.append(blob)
    return np.vstack(pts)


def loops_cloud(n, n_loops, radius=1.0, noise=0.05, outlier_fraction=0.0, rng=None):
    """Cloud of ``n_loops`` noisy circles arranged on a big circle.

    Each loop is a circle of its own radius; loop centers sit at equally spaced
    angles on a big circle whose radius guarantees the loops stay separated
    (adjacent centers at least 4 x the largest loop radius apart), so the alpha
    complex of the whole cloud has one prominent ``H_1`` feature per loop with
    persistence approximately equal to that loop's radius. A fraction
    ``outlier_fraction`` of the points are drawn uniformly over the bounding
    box of the arrangement (topological clutter).

    Args:
        n: total number of points.
        n_loops: number of loops (prominent ``H_1`` features).
        radius: loop radius; either a scalar (all loops equal) or a length
            ``n_loops`` array of per-loop radii.
        noise: standard deviation of the isotropic Gaussian jitter.
        outlier_fraction: fraction of points placed uniformly in the bounding
            box (default 0.0).
        rng: seed or Generator.

    Returns:
        ``(n, 2)`` point cloud.
    """
    rng = _as_rng(rng)
    n_loops = int(n_loops)
    if n_loops < 1:
        raise ValueError("n_loops must be >= 1")
    radii = np.full(n_loops, float(radius)) if np.isscalar(radius) else np.asarray(radius, dtype=float)
    if radii.shape[0] != n_loops:
        raise ValueError("radius array must have length n_loops")
    max_r = float(radii.max())
    n_outliers = int(round(outlier_fraction * n))
    n_loop_pts = n - n_outliers
    per_loop, remainder = divmod(n_loop_pts, n_loops)
    if per_loop < 8:
        raise ValueError("n too small for n_loops (need >= 8 points per loop)")

    big_R = 2.5 * max_r if n_loops > 1 else 0.0
    angles = 2.0 * np.pi * np.arange(n_loops) / n_loops
    centers = big_R * np.column_stack([np.cos(angles), np.sin(angles)])

    pts = []
    for k in range(n_loops):
        size = per_loop + (1 if k < remainder else 0)
        theta = rng.uniform(0.0, 2.0 * np.pi, size=size)
        circle = centers[k] + radii[k] * np.column_stack([np.cos(theta), np.sin(theta)])
        pts.append(circle + rng.normal(scale=noise, size=circle.shape))
    if n_outliers > 0:
        span = big_R + max_r
        pts.append(rng.uniform(-span, span, size=(n_outliers, 2)))
    return np.vstack(pts)

In [ ]:
%%writefile tda2s/dgp/simulation.py
"""Covariate-driven point-cloud DGP harness (Phase 0.6).

The harness generates *datasets* of point clouds for two-sample topological
testing. It exposes independently dialable knobs:

  * covariates ``X ~ N(0, I)`` or a two-component Gaussian mixture;
  * propensity ``pi(X) = expit(prop_scale * X @ beta)`` (imbalance dialled via
    ``|beta|`` and ``prop_scale``);
  * covariate-driven topology: a deterministic function ``topology_knob(x) ->
    (n_loops, radius, noise)`` maps each covariate row to the generator
    parameters of its cloud;
  * an optional direct group effect that shifts the topology of group A
    regardless of ``X`` (power experiments).

Key design property: with ``group_effect=0`` the topology of a cloud is a
deterministic function of its covariate vector ``X_i``, so conditional on ``X``
the two groups have the *identical* topological law; only the propensity
differs between groups. This is the exact structure Phase 2's covariate-shift
gate experiments require. With ``group_effect > 0`` the group-A loop count is
shifted regardless of ``X``.

The harness records, per cloud, the true generator parameters (``n_loops``,
per-loop ``radii``, ``noise``, ``outlier_fraction``) in ``CloudSample.oracle``,
so tests and experiments can verify that dialled knobs are recovered from the
oracle.

``to_silhouette_sample`` converts a sample of clouds to the observed triplet
``(phi, A, X)`` in the ``tcda_uq`` silhouette convention (``phi`` of shape
``[n, n_hom_dim, resolution]``), so Phase 3 can feed both harnesses through the
same downstream estimators.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.special import expit

from .clouds import loops_cloud

_MU1 = np.array([1.0, 0.6, -0.7, 2.2, -1.0])
_MU2 = np.array([0.4, -0.4, -0.6, 3.3, 3.0])
_BETA = np.array([-0.5, -0.1, 0.6, 0.1, 0.1])


@dataclass
class CloudSample:
    """One realised dataset of point clouds with per-cloud topology oracles.

    Attributes:
        clouds: list of ``(m_i, 2)`` point clouds, one per unit.
        X: ``[n, d_x]`` covariate matrix.
        A: ``[n]`` group labels (0 = B, 1 = A).
        propensity: ``[n]`` true propensity ``pi(X)``.
        oracle: dict mapping unit index to ``{"n_loops", "radii", "noise",
            "outlier_fraction"}``, the exact generator parameters used.
    """

    clouds: list
    X: np.ndarray
    A: np.ndarray
    propensity: np.ndarray
    oracle: dict

    @property
    def true_n_loops(self) -> np.ndarray:
        """``[n]`` oracle loop counts."""
        return np.array([self.oracle[i]["n_loops"] for i in range(len(self.clouds))])

    @property
    def true_radii(self) -> list:
        """Per-cloud oracle loop radii (list of per-loop arrays)."""
        return [self.oracle[i]["radii"] for i in range(len(self.clouds))]

    @property
    def true_noise(self) -> np.ndarray:
        """``[n]`` oracle point jitter scales."""
        return np.array([self.oracle[i]["noise"] for i in range(len(self.clouds))])

    def observed(self, **sil_kwargs):
        """Observed triplet ``(phi, A, X)`` in tcda_uq silhouette format."""
        return to_silhouette_sample(self.clouds, self.X, self.A, **sil_kwargs)


def _default_topology_knob(gamma, k_max, radius, noise):
    """Default knob: ``n_loops = 1 + floor(expit(gamma * x0) * k_max)``."""

    def knob(x):
        n_loops = 1 + int(np.floor(expit(gamma * x[0]) * k_max))
        return n_loops, radius, noise

    return knob


class CloudSampleDGP:
    """Two-group point-cloud DGP with covariate-driven topology.

    Args:
        n_per_group: half the dataset size; ``n = 2 * n_per_group`` clouds are
            drawn in total. Labels are then drawn as ``A_i ~ Bern(pi(X_i))``, so
            the two groups are *not* forced to be equal-sized -- that imbalance
            is exactly the confounding this harness exists to create. Pass
            ``beta=np.zeros(d_x)`` for a balanced, unconfounded design.
        m: number of points per cloud.
        d_x: covariate dimension.
        covariate: ``"gaussian"`` (standard normal) or ``"mixture"``
            (two-component Gaussian mixture, tcda_uq conventions).
        beta: propensity coefficients (``None`` uses the tcda_uq default).
        prop_scale: multiplier of the propensity logit (``> 1`` pushes
            ``pi(X)`` toward ``{0, 1}``).
        topology_knob: callable ``x -> (n_loops, radius, noise)`` mapping one
            covariate row to generator parameters. ``None`` uses
            ``n_loops = 1 + floor(expit(gamma * x[0]) * k_max)`` with fixed
            ``radius`` and ``noise``. A deterministic knob is what makes the
            two groups conditionally identical given ``X``.
        gamma, k_max: slope and max extra loops of the default knob.
        radius, noise, outlier_fraction: fixed generator parameters used when
            ``topology_knob`` is ``None``.
        group_effect: integer loop-count shift applied to group A regardless
            of ``X`` (0 = conditionally identical groups).
        seed: recorded on the instance for provenance; the model coefficients
            (mixture means, default ``beta``) are fixed constants, so sampling
            randomness is controlled entirely by ``sample(rng=...)``.
    """

    def __init__(
        self,
        n_per_group: int = 25,
        m: int = 120,
        d_x: int = 3,
        covariate: str = "gaussian",
        beta=None,
        prop_scale: float = 1.0,
        topology_knob=None,
        gamma: float = 1.0,
        k_max: int = 3,
        radius: float = 1.0,
        noise: float = 0.05,
        outlier_fraction: float = 0.0,
        group_effect: int = 0,
        seed: int = 0,
    ):
        if n_per_group < 1:
            raise ValueError("n_per_group must be >= 1")
        if m < 40:
            raise ValueError("m must be >= 40 to keep loop persistence recoverable")
        if covariate not in ("gaussian", "mixture"):
            raise ValueError("covariate must be 'gaussian' or 'mixture'")

        self.n_per_group = int(n_per_group)
        self.m = int(m)
        self.d_x = int(d_x)
        self.covariate = covariate
        self.prop_scale = float(prop_scale)
        self.gamma = float(gamma)
        self.k_max = int(k_max)
        self.radius = float(radius)
        self.noise = float(noise)
        self.outlier_fraction = float(outlier_fraction)
        self.group_effect = int(group_effect)

        self.seed = seed
        self.mu1 = _MU1[: self.d_x]
        self.mu2 = _MU2[: self.d_x]
        self.Sigma = np.eye(self.d_x) * 0.5
        self.beta = np.asarray(beta if beta is not None else _BETA[: self.d_x], dtype=float)
        if self.beta.shape[0] != self.d_x:
            raise ValueError("beta must have length d_x")

        self.topology_knob = topology_knob if topology_knob is not None else _default_topology_knob(
            self.gamma, self.k_max, self.radius, self.noise
        )
        self._max_loops = max(1, self.m // 8)

    def propensity(self, X):
        """True propensity ``pi(X) = expit(prop_scale * X @ beta)``, ``[n]``."""
        return expit(self.prop_scale * (np.asarray(X, dtype=float) @ self.beta))

    def topology(self, x):
        """Topology tuple ``(n_loops, radius, noise)`` for one covariate row."""
        return self.topology_knob(np.asarray(x, dtype=float))

    def _sample_covariates(self, n, rng):
        if self.covariate == "gaussian":
            return rng.normal(size=(n, self.d_x))
        n1 = n // 2
        X1 = rng.multivariate_normal(self.mu1, self.Sigma, size=n1)
        X2 = rng.multivariate_normal(self.mu2, self.Sigma, size=n - n1)
        return np.vstack([X1, X2])

    def sample(self, n_per_group=None, X=None, rng=None) -> CloudSample:
        """Draw a dataset of ``2 * n_per_group`` point clouds.

        Args:
            n_per_group: overrides the constructor default.
            X: optional fixed covariate matrix ``[n, d_x]`` (e.g. repeated rows
                for conditional-identical-group checks); drawn otherwise.
            rng: seed or Generator.

        Returns:
            :class:`CloudSample` with ``clouds``, ``X``, ``A``, ``propensity``
            and the per-cloud ``oracle`` of true generator parameters.
        """
        rng = rng if isinstance(rng, np.random.Generator) else np.random.default_rng(rng)
        n_per_group = self.n_per_group if n_per_group is None else int(n_per_group)
        n = 2 * n_per_group
        if X is None:
            X = self._sample_covariates(n, rng)
        else:
            X = np.asarray(X, dtype=float)
            if X.shape != (n, self.d_x):
                raise ValueError(f"X must have shape ({n}, {self.d_x})")

        pi = self.propensity(X)
        A = rng.binomial(1, pi).astype(int)

        clouds, oracle = [], {}
        for i in range(n):
            n_loops, radius, noise = self.topology(X[i])
            if A[i] == 1:
                n_loops = int(n_loops) + self.group_effect
            n_loops = int(np.clip(n_loops, 1, self._max_loops))
            radii = np.full(n_loops, float(radius))
            cloud = loops_cloud(self.m, n_loops, radius=radii, noise=noise,
                                outlier_fraction=self.outlier_fraction, rng=rng)
            clouds.append(cloud)
            oracle[i] = {
                "n_loops": n_loops,
                "radii": radii.copy(),
                "noise": float(noise),
                "outlier_fraction": self.outlier_fraction,
            }
        return CloudSample(clouds=clouds, X=X, A=A, propensity=pi, oracle=oracle)


# --- Phase 2.2 masking DGP (exact Simpson cancellation) -----------------------
# Three covariate strata X in {0, 1, 2} (uniform), propensity e = (1/2, 1/2, 1/4).
# Conditional diagram laws: strata 0, 1 have identical laws in both arms; the
# treatment effect lives entirely in stratum 2, where the treated law L_C and
# the control law (4/15)(L_A + L_B) + (7/15)L_C are chosen so that the marginal
# observational laws coincide exactly:
#
#     L(D|A=1) = (2/5)(L_A + L_B) + (1/5)L_C  =  L(D|A=0).
#
# This is exact (verified in tests/test_phase2.py): every valid level-alpha
# permutation test of H0^cond has power exactly alpha there, while the
# covariate-standardized topological effect is
#
#     psi_d = (1/3)(8/15)(m_C - (m_A + m_B) / 2) != 0
#
# whenever the mean silhouettes of the three types do not satisfy m_C =
# (m_A + m_B) / 2 (here m_A = Lambda_{r_A}, m_B = Lambda_{r_B}, m_C = Lambda_{r_C}
# on the persistence scale ~0.75 * radius, with r_A = 1, r_B = 2, r_C = 4).

_MASK_E = np.array([0.5, 0.5, 0.25])          # per-stratum propensity
_MASK_W0 = 4.0 / 15.0                          # control-mixture weights at stratum 2
_MASK_W1 = 7.0 / 15.0


def masking_stratum_sample(n_per_group, m=120, noise=0.05, radius_a=1.0,
                           radius_b=2.0, radius_c=4.0, seed=None):
    """Exact Simpson-masking DGP: H0^cond true, H0^out false (Phase 2.2).

    Stratum 0 and 1 units are 1-loop clouds of radius ``radius_a`` / ``radius_b``
    in *both* arms (no treatment effect there). Stratum 2 treated units are
    2-loop clouds of radius ``radius_c``; stratum 2 control units draw a cloud
    type from the mixture ``(4/15, 4/15, 7/15)`` over the three types. With
    propensity e = (1/2, 1/2, 1/4) the marginal observational diagram laws
    coincide exactly between arms, while the covariate-standardized silhouette
    effect is non-zero.

    Args:
        n_per_group: number of units per arm of the sample.
        m: points per cloud.
        noise: per-cloud Gaussian jitter scale.
        radius_a, radius_b, radius_c: loop radii of the three cloud types.
        seed: RNG seed.

    Returns:
        :class:`CloudSample` with ``X`` in {0, 1, 2}, the true propensity, and
        an oracle recording per-unit (stratum, cloud type, radii).
    """
    rng = np.random.default_rng(seed)
    n = 2 * int(n_per_group)
    X = np.tile(np.arange(3), int(np.ceil(n / 3)))[:n]
    e = _MASK_E[X]
    A = rng.binomial(1, e).astype(int)

    def _cloud(radius, n_loops):
        return loops_cloud(m, n_loops, radius=radius, noise=noise, rng=rng)

    clouds, oracle = [], {}
    for i in range(n):
        x, a = int(X[i]), int(A[i])
        if x in (0, 1):
            radius = radius_a if x == 0 else radius_b
            n_loops = 1
        else:  # stratum 2: the only stratum carrying the effect
            if a == 1:
                radius, n_loops = radius_c, 2
            else:
                u = rng.random()
                if u < _MASK_W0:
                    radius, n_loops = radius_a, 1
                elif u < 2 * _MASK_W0:
                    radius, n_loops = radius_b, 1
                else:
                    radius, n_loops = radius_c, 2
        clouds.append(_cloud(radius, n_loops))
        oracle[i] = {"stratum": x, "n_loops": n_loops,
                     "radii": np.full(n_loops, float(radius)), "noise": noise}
    return CloudSample(clouds=clouds, X=np.asarray(X, dtype=float).reshape(-1, 1),
                       A=A, propensity=e, oracle=oracle)


def _silhouette_from_diagrams(diags, interval, r, resolution):
    """Power-weighted silhouette of a diagram list (tcda_uq convention).

    Delegates to ``tda2s.vec.silhouette``, which uses the same power-weight
    convention (``|death - birth| ** r``, ``keep_endpoints=True``) as
    ``tcda_uq.silhouette.compute_silhouette``.
    """
    from tda2s.vec import silhouette

    return silhouette(diags, interval=interval, r=r, resolution=resolution)


def to_silhouette_sample(clouds, X, A, filtration="alpha", homology_dims=(0, 1),
                         interval=(0.0, 1.0), r=3.0, resolution=100, **ph_kwargs):
    """Convert clouds to the observed ``(phi, A, X)`` silhouette triplet.

    Each cloud is mapped to persistence diagrams via
    ``tda2s.ph.compute_diagrams`` and then to a power-weighted silhouette, so
    the output matches ``tcda_uq``'s observed format: ``phi`` has shape
    ``[n, n_hom_dim, resolution]``, ``A`` is ``[n]`` and ``X`` is ``[n, d_x]``.

    Args:
        clouds: iterable of ``(m_i, 2)`` point clouds.
        X: ``[n, d_x]`` covariate matrix.
        A: ``[n]`` group labels.
        filtration, homology_dims: passed to ``tda2s.ph.compute_diagrams``.
        interval, r, resolution: silhouette domain, power-weight exponent and
            grid size (tcda_uq conventions).
        **ph_kwargs: extra ``compute_diagrams`` keyword arguments.

    Returns:
        Tuple ``(phi, A, X)`` with ``phi`` of shape ``[n, n_hom_dim, resolution]``.
    """
    from tda2s.ph import compute_diagrams

    n_hom = len(homology_dims)
    n = len(clouds)
    phi = np.empty((n, n_hom, resolution))
    for i, cloud in enumerate(clouds):
        diags = compute_diagrams(cloud, filtration=filtration,
                                 homology_dims=homology_dims, **ph_kwargs)
        phi[i] = _silhouette_from_diagrams(diags, interval=interval, r=r,
                                           resolution=resolution)
    return phi, np.asarray(A, dtype=int), np.asarray(X, dtype=float)

In [ ]:
%%writefile tda2s/ph/__init__.py
"""PH pipeline: point cloud -> persistence diagrams, uniform API.

Filtrations: VR (gudhi), ripser (fast), Alpha, Cech, cubical-sublevel (grid
distance transform), DTM-Rips (weighted Rips with DTM vertex weights).

Conventions
-----------
* A diagram is a ``(k, 2)`` float array of (birth, death) pairs, one per
  homology dimension, returned as a list indexed by homology dim.
* Essential classes (death = inf) are dropped: with filtrations of compact
  point sets every class dies, and tcda_uq uses the same convention.
* All filtration values are radii. Alpha *and* Delaunay-Cech report squared
  circumradii internally, so both extractors take a square root; Rips/ripser
  and DTM-Rips are already on the radius scale.
* Diagrams are cached to disk keyed by (point-cloud hash, filtration, params):
  permutation tests must never recompute PH inside the permutation loop.
* gudhi pair format: ``st.persistence()`` yields ``(dim, (birth, death))``
  tuples with essential classes as ``(dim, (birth, inf))``. Extractors are
  defensive about non-tuple payloads anyway (see ``_drop_infinite``), and
  infinite deaths are always dropped, so downstream code never sees infs.
* ``dtm-rips`` with ``homology_dims`` including 2 and an unbounded
  ``max_edge_length`` enumerates every 3-simplex of the cloud (combinatorial
  blow-up); pass a bounded ``max_edge_length`` for that configuration.
"""
from __future__ import annotations

import hashlib
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

_HOMOLOGY_DIMS = (0, 1, 2)


@dataclass
class PhParams:
    """Filtration parameters (cached under the hash of these + the cloud)."""

    filtration: str = "alpha"
    homology_dims: Tuple[int, ...] = _HOMOLOGY_DIMS
    max_edge_length: Optional[float] = None
    grid_size: int = 64
    dtm_k: int = 20
    cache_dir: Optional[str] = None

    def key(self, points: np.ndarray) -> str:
        h = hashlib.sha256()
        h.update(np.ascontiguousarray(points, dtype=np.float32).tobytes())
        h.update(repr((self.filtration, self.homology_dims, self.max_edge_length,
                       self.grid_size, self.dtm_k)).encode())
        return h.hexdigest()[:24]


def _points_to_float(points) -> np.ndarray:
    pts = np.asarray(points, dtype=float)
    if pts.ndim != 2:
        raise ValueError(f"points must be (m, d), got {pts.shape}")
    return pts


def _drop_infinite(dgm: np.ndarray) -> np.ndarray:
    dgm = np.asarray(dgm, dtype=float)
    if dgm.ndim != 2:
        return dgm.reshape(0, 2)
    if dgm.size == 0:
        return dgm.reshape(0, 2)
    return dgm[np.isfinite(dgm[:, 1])]


def _alpha_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import gudhi as gd

    alpha = gd.AlphaComplex(points=pts)
    st = alpha.create_simplex_tree()
    if max_edge_length is not None:
        st.prune_above_filtration(max_edge_length**2)
    st.compute_persistence()
    out = []
    for d in range(max_dim + 1):
        finite = [p for p in st.persistence() if p[0] == d
                  and isinstance(p[1], tuple) and np.isfinite(p[1][1])]
        dgm = np.array([p[1] for p in finite], dtype=float).reshape(-1, 2)
        out.append(np.sqrt(dgm) if dgm.size else dgm)
    return out


def _vr_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import gudhi as gd

    rc = gd.RipsComplex(points=pts, max_edge_length=max_edge_length or np.inf)
    st = rc.create_simplex_tree(max_dimension=max_dim + 1)
    st.compute_persistence()
    return [_drop_infinite(np.array(
        [p[1] for p in st.persistence() if p[0] == d], dtype=float).reshape(-1, 2))
        for d in range(max_dim + 1)]


def _ripser_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import ripser

    dgms = ripser.ripser(pts, maxdim=max_dim, thresh=float(max_edge_length) if max_edge_length else np.inf)["dgms"]
    out = []
    for d in range(max_dim + 1):
        dgm = _drop_infinite(np.asarray(dgms[d], dtype=float))
        out.append(dgm)
    return out


def _cech_diagrams(pts: np.ndarray, max_dim: int, max_edge_length) -> List[np.ndarray]:
    import gudhi as gd

    cc = gd.DelaunayCechComplex(points=pts)
    st = cc.create_simplex_tree()
    if max_edge_length is not None:
        st.prune_above_filtration(max_edge_length**2)
    st.compute_persistence()
    out = []
    for d in range(max_dim + 1):
        dgm = _drop_infinite(np.array(
            [p[1] for p in st.persistence() if p[0] == d], dtype=float).reshape(-1, 2))
        # DelaunayCechComplex reports *squared* circumradii (same convention as
        # AlphaComplex); take the square root so every filtration in this module
        # is on the radius scale.
        out.append(np.sqrt(dgm) if dgm.size else dgm)
    return out


def _cubical_diagrams(pts: np.ndarray, max_dim: int, grid_size: int) -> List[np.ndarray]:
    """Cubical sublevel filtration of the distance-to-cloud function on a grid.

    The grid distance transform is a piecewise-Lipschitz proxy for the
    distance function; its sublevel sets reproduce the cloud's topology at
    scales above the grid resolution.
    """
    import gudhi as gd
    from scipy.spatial import cKDTree

    # Pad relative to the cloud's extent, not by an absolute epsilon: an
    # absolute pad would make the grid (and hence the filtration) depend on the
    # cloud's units, breaking scale equivariance at the 1e-6 level.
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    pad = 1e-6 * float(np.max(hi - lo))
    pad = pad if pad > 0 else 1e-6
    lo, hi = lo - pad, hi + pad
    axes = [np.linspace(lo[j], hi[j], grid_size) for j in range(pts.shape[1])]
    grid = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(-1, pts.shape[1])
    dist = cKDTree(pts).query(grid, k=1)[0].reshape([grid_size] * pts.shape[1])
    cc = gd.CubicalComplex(dimensions=list(dist.shape), top_dimensional_cells=dist.ravel())
    cc.compute_persistence()
    return [_drop_infinite(np.array(cc.persistence_intervals_in_dimension(d), dtype=float).reshape(-1, 2))
            for d in range(min(max_dim, 2) + 1)]


def _dtm_rips_diagrams(pts: np.ndarray, max_dim: int, dtm_k: int, max_edge_length) -> List[np.ndarray]:
    """Weighted Rips with DTM vertex weights (Anai et al. 2019 construction)."""
    import numpy as np
    from scipy.spatial.distance import cdist
    from gudhi.point_cloud.dtm import DistanceToMeasure
    from gudhi.weighted_rips_complex import WeightedRipsComplex

    # DistanceToMeasure is not callable in gudhi 3.11 (dtm(pts) raises
    # TypeError); transform after fit.
    dtm = DistanceToMeasure(k=dtm_k)
    weights = np.asarray(dtm.fit_transform(pts), dtype=float)
    # gudhi's WeightedRipsComplex expects a pairwise distance matrix.
    dist = cdist(pts, pts)
    rc = WeightedRipsComplex(distance_matrix=dist, weights=weights,
                             max_filtration=max_edge_length if max_edge_length is not None else np.inf)
    st = rc.create_simplex_tree(max_dimension=max_dim + 1)
    st.compute_persistence()
    return [_drop_infinite(np.array(
        [p[1] for p in st.persistence() if p[0] == d], dtype=float).reshape(-1, 2))
        for d in range(max_dim + 1)]


def compute_diagrams(points, filtration: str = "alpha",
                     homology_dims: Sequence[int] = _HOMOLOGY_DIMS,
                     max_edge_length: Optional[float] = None,
                     grid_size: int = 64, dtm_k: int = 20,
                     standardise: Optional[Tuple[np.ndarray, np.ndarray]] = None,
                     cache_dir: Optional[str] = None) -> List[np.ndarray]:
    """Compute persistence diagrams of a point cloud under ``filtration``.

    Args:
        points: ``(m, d)`` point cloud.
        filtration: one of {"vr", "ripser", "alpha", "cech", "cubical", "dtm-rips"}.
        homology_dims: homology dimensions to keep.
        max_edge_length: filtration cutoff (radius units; None = unbounded).
        grid_size: grid edge length for "cubical".
        dtm_k: DTM neighbourhood size for "dtm-rips".
        standardise: optional ``(mean, scale)`` pair of ``(d,)`` arrays applied
            to the points as ``(points - mean) / scale`` before filtration.
            Externally supplied by the caller (e.g. a fixed study-region
            background) instead of per-cloud scaling, so diagrams of different
            clouds live in one shared coordinate system.
        cache_dir: if set, diagrams are cached/loaded keyed by cloud+params hash.

    Returns:
        List of ``(k, 2)`` (birth, death) arrays, indexed by homology dim.
    """
    pts = _points_to_float(points)
    if standardise is not None:
        mean, scale = standardise
        pts = (pts - np.asarray(mean, dtype=float)) / np.asarray(scale, dtype=float)
    dims = tuple(int(d) for d in homology_dims)
    max_dim = max(dims)
    params = PhParams(filtration=filtration, homology_dims=dims,
                      max_edge_length=max_edge_length, grid_size=grid_size,
                      dtm_k=dtm_k, cache_dir=cache_dir)
    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)
        path = os.path.join(cache_dir, f"{params.key(pts)}.npz")
        if os.path.exists(path):
            with np.load(path, allow_pickle=False) as z:
                return [z[f"d{d}"] for d in dims]

    if filtration == "alpha":
        all_d = _alpha_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "vr":
        all_d = _vr_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "ripser":
        all_d = _ripser_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "cech":
        all_d = _cech_diagrams(pts, max_dim, max_edge_length)
    elif filtration == "cubical":
        all_d = _cubical_diagrams(pts, max_dim, grid_size)
    elif filtration == "dtm-rips":
        all_d = _dtm_rips_diagrams(pts, max_dim, dtm_k, max_edge_length)
    else:
        raise ValueError(f"unknown filtration: {filtration}")

    out = [all_d[d] for d in dims]
    if cache_dir:
        np.savez(path, **{f"d{d}": arr for d, arr in zip(dims, out)})
    return out


def betti_numbers(diags: Sequence[np.ndarray], persistence_threshold: float) -> List[int]:
    """Count features with persistence strictly above a threshold, per dim.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays (see ``compute_diagrams``).
        persistence_threshold: features with ``death - birth > threshold`` count.

    Returns:
        One count per homology dim.
    """
    counts = []
    for dgm in diags:
        dgm = np.asarray(dgm, dtype=float).reshape(-1, 2)
        counts.append(int((dgm[:, 1] - dgm[:, 0] > persistence_threshold).sum()))
    return counts

In [ ]:
%%writefile tda2s/vec/__init__.py
"""Vectorisation stack: persistence diagrams -> fixed-size feature vectors.

Uniform entry point ``vectorise(diags, representation, **kwargs)`` dispatches
to per-representation functions. Representations:

* ``silhouette``: power-weighted persistence silhouette (gudhi
  ``representations.Silhouette``), same parameterisation as
  ``tcda_uq.silhouette.core.compute_silhouette`` (interval (0.0, 0.2),
  resolution 100, keep_endpoints=True, power r=3).
* ``landscape``: persistence landscapes (gudhi ``representations.Landscape``).
* ``betti``: Betti curves ``B_d(t) = #{features: birth <= t < death}`` on a
  t-grid (implemented locally).
* ``euler``: Euler curves ``sum_d (-1)^d B_d(t)``.
* ``image``: persistence images (gudhi ``representations.PersistenceImage``).
* ``measure``: persistence measure (Divol-Lacombe style): the diagram is a
  measure ``mu = sum_p w_p * delta_{(b, (b+d)/2)}`` over the (birth, mid)
  plane, projected onto a fixed 2-D grid of bins.

Conventions
-----------
* ``diags``: list of ``(k, 2)`` (birth, death) arrays, one per homology dim
  (same format as ``tda2s.ph.compute_diagrams``); deaths must be finite
  (essential classes are dropped upstream).
* Each homology dim is vectorised separately, so outputs carry a leading
  per-dim axis; all outputs are float64 arrays.
* ``interval`` (or ``sample_range``) defaults, when not given, to ``(0.0,
  max death)`` derived from the diagrams.
"""
from __future__ import annotations

from typing import Callable, List, Optional, Sequence, Tuple

import numpy as np


def _diagram_list(diags: Sequence[np.ndarray]) -> List[np.ndarray]:
    """Coerce the per-dim diagram list to a list of (k, 2) float arrays."""
    out = []
    for dgm in diags:
        out.append(np.asarray(dgm, dtype=float).reshape(-1, 2))
    return out


def _default_interval(diags: Sequence[np.ndarray]) -> Tuple[float, float]:
    """Default sample range ``(0, max death)`` across all dims."""
    hi = 1.0
    for dgm in _diagram_list(diags):
        if len(dgm):
            hi = max(hi, float(dgm[:, 1].max()))
    return (0.0, hi)


def _power_weight(point: np.ndarray, r: float) -> float:
    """Power weight ``|death - birth|**r`` of a persistence point."""
    return float(np.abs(point[1] - point[0]) ** r)


def silhouette(diags: Sequence[np.ndarray], interval=(0.0, 0.2), r: float = 3.0,
               resolution: int = 100) -> np.ndarray:
    """Power-weighted persistence silhouette.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        interval: sample range ``[t_min, t_max]`` of the silhouette.
        r: power-weight exponent ``w = (death - birth)**r``.
        resolution: number of grid points.

    Returns:
        ``(n_dims, resolution)`` array of silhouette values.
    """
    from gudhi.representations import Silhouette

    s = Silhouette(weight=lambda x: _power_weight(x, r), resolution=resolution,
                   sample_range=list(interval), keep_endpoints=True)
    return np.asarray(s.fit_transform(_diagram_list(diags)), dtype=float)


def landscape(diags: Sequence[np.ndarray], num_landscapes: int = 5,
              resolution: int = 100, interval: Optional[Tuple[float, float]] = None) -> np.ndarray:
    """Persistence landscapes of each dim's diagram.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        num_landscapes: number of landscape functions per dim.
        resolution: number of grid points per landscape.
        interval: sample range; defaults to ``(0, max death)``.

    Returns:
        ``(n_dims, num_landscapes, resolution)`` array of landscape values.
    """
    from gudhi.representations import Landscape

    iv = interval if interval is not None else _default_interval(diags)
    l = Landscape(num_landscapes=num_landscapes, resolution=resolution,
                  sample_range=list(iv))
    out = np.asarray(l.fit_transform(_diagram_list(diags)), dtype=float)
    return out.reshape(len(diags), num_landscapes, resolution)


def betti_curve(diags: Sequence[np.ndarray], interval: Optional[Tuple[float, float]] = None,
                n_points: int = 100) -> np.ndarray:
    """Betti curves ``B_d(t) = #{features: birth <= t < death}``.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        interval: t-grid range; defaults to ``(0, max death)``.
        n_points: number of t-grid points.

    Returns:
        ``(n_dims, n_points)`` array of Betti numbers on the t-grid.
    """
    iv = interval if interval is not None else _default_interval(diags)
    grid = np.linspace(iv[0], iv[1], n_points)
    rows = []
    for dgm in _diagram_list(diags):
        if len(dgm) == 0:
            rows.append(np.zeros(n_points))
            continue
        alive = (dgm[:, 0][:, None] <= grid[None, :]) & (grid[None, :] < dgm[:, 1][:, None])
        rows.append(alive.sum(axis=0, dtype=np.int64).astype(float))
    return np.stack(rows)


def euler_curve(diags: Sequence[np.ndarray], interval: Optional[Tuple[float, float]] = None,
                n_points: int = 100) -> np.ndarray:
    """Euler characteristic curve ``sum_d (-1)^d B_d(t)``.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        interval: t-grid range; defaults to ``(0, max death)``.
        n_points: number of t-grid points.

    Returns:
        ``(n_points,)`` array of Euler characteristics on the t-grid.
    """
    b = betti_curve(diags, interval=interval, n_points=n_points)
    signs = np.array([(-1.0) ** d for d in range(len(diags))])
    return signs @ b


def persistence_image(diags: Sequence[np.ndarray], bandwidth: float = 0.1,
                      weight: Optional[Callable[[np.ndarray], float]] = None,
                      resolution=(10, 10),
                      interval: Optional[Tuple[float, float]] = None) -> np.ndarray:
    """Persistence images (gaussian kernels centred on diagram points).

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        bandwidth: gaussian kernel width.
        weight: point weight function; defaults to persistence ``death - birth``.
        resolution: ``(n_pixels_x, n_pixels_y)`` grid.
        interval: ``[t_min, t_max]`` shared by both axes; defaults to
            ``(0, max death)``.

    Returns:
        ``(n_dims, n_pixels_x, n_pixels_y)`` array of image intensities.
    """
    from gudhi.representations import PersistenceImage

    iv = interval if interval is not None else _default_interval(diags)
    w = weight if weight is not None else lambda x: x[1] - x[0]
    pi = PersistenceImage(bandwidth=bandwidth, weight=w,
                          resolution=list(resolution),
                          im_range=[iv[0], iv[1], iv[0], iv[1]])
    out = np.asarray(pi.fit_transform(_diagram_list(diags)), dtype=float)
    return out.reshape(len(diags), int(resolution[0]), int(resolution[1]))


def persistence_measure(diags: Sequence[np.ndarray],
                        weight: Optional[Callable[[np.ndarray], float]] = None,
                        interval: Optional[Tuple[float, float]] = None,
                        n_bins: int = 32) -> np.ndarray:
    """Persistence measure: ``mu = sum_p w_p * delta_{(b, (b+d)/2)}``.

    The diagram is represented as a weighted point measure on the (birth,
    mid) plane with ``mid = (b + d) / 2`` (Divol-Lacombe coordinates),
    projected onto a fixed regular grid of bins as a weighted 2-D histogram.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        weight: point weight function; defaults to persistence ``death - birth``.
        interval: shared ``[t_min, t_max]`` for both the birth and mid axes;
            defaults to ``(0, max death)``.
        n_bins: number of bins along each axis.

    Returns:
        ``(n_dims, n_bins, n_bins)`` array of aggregated weights per bin.
    """
    iv = interval if interval is not None else _default_interval(diags)
    w = weight if weight is not None else lambda x: x[1] - x[0]
    edges = np.linspace(iv[0], iv[1], n_bins + 1)
    out = []
    for dgm in _diagram_list(diags):
        if len(dgm) == 0:
            out.append(np.zeros((n_bins, n_bins)))
            continue
        mid = (dgm[:, 0] + dgm[:, 1]) / 2.0
        ws = np.array([w(p) for p in dgm], dtype=float)
        hist, _, _ = np.histogram2d(dgm[:, 0], mid, bins=[edges, edges], weights=ws)
        out.append(hist)
    return np.stack(out)


def vectorise(diags: Sequence[np.ndarray], representation: str, **kwargs) -> np.ndarray:
    """Vectorise persistence diagrams under a named representation.

    Args:
        diags: list of ``(k, 2)`` (birth, death) arrays, one per homology dim.
        representation: one of {"silhouette", "landscape", "betti", "euler",
            "image", "measure"}.
        **kwargs: passed to the per-representation function (e.g. ``interval``,
            ``resolution``, ``r``).

    Returns:
        The representation vector; see the individual functions for shapes.
    """
    if representation == "silhouette":
        return silhouette(diags, **kwargs)
    if representation == "landscape":
        return landscape(diags, **kwargs)
    if representation == "betti":
        return betti_curve(diags, **kwargs)
    if representation == "euler":
        return euler_curve(diags, **kwargs)
    if representation == "image":
        return persistence_image(diags, **kwargs)
    if representation == "measure":
        return persistence_measure(diags, **kwargs)
    raise ValueError(
        f"unknown representation {representation!r}; expected one of "
        "['silhouette', 'landscape', 'betti', 'euler', 'image', 'measure']")


In [ ]:
%%writefile tda2s/resample/__init__.py
"""Resampling engine: null distributions for the Phase 3-5 tests.

Schemes
-------
* ``permutation_test`` -- label permutation (exact null under exchangeability),
  optionally restricted to permutation *within propensity strata* (the
  covariate-preserving variant).
* ``multiplier_bootstrap`` -- Gaussian/Rademacher multiplier bootstrap over a
  per-unit influence-function matrix, calibrated to the weak limit of the
  empirical process: null draws are ``sqrt(n) * sup_t | n^{-1/2} sum_i g_i
  inf_i(t) |``.
* ``paired_bootstrap`` -- unit resampling with replacement preserving the
  treated/control split (paired designs, two-sample mean differences).
* ``smoothed_bootstrap`` -- Roycraft-Krebs-Polonik smoothed bootstrap for
  persistent Betti numbers: resample diagrams with replacement, jitter
  (birth, death) coordinates by N(0, sigma^2), recompute the statistic.
* ``cross_fit_folds`` -- k-fold cross-fitting index splits.
* ``p_value`` -- Monte Carlo p-value helper.
"""
from __future__ import annotations

from typing import Callable, List, Optional, Sequence, Tuple

import numpy as np

from .smoothing import betti_curve

__all__ = [
    "permutation_test",
    "multiplier_bootstrap",
    "paired_bootstrap",
    "smoothed_bootstrap",
    "cross_fit_folds",
    "p_value",
]


def p_value(observed: float, null_stats: Sequence[float], alternative: str = "greater") -> float:
    """Monte Carlo p-value with Phipson-Smyth correction.

    ``alternative="greater"``: P(T* >= T_obs); ``"less"``: P(T* <= T_obs);
    ``"two-sided"``: P(|T*| >= |T_obs|) with the same correction.
    """
    null_stats = np.asarray(null_stats, dtype=float)
    if null_stats.size == 0:
        return 1.0
    if alternative == "greater":
        return (1.0 + (null_stats >= observed).sum()) / (1.0 + null_stats.size)
    if alternative == "less":
        return (1.0 + (null_stats <= observed).sum()) / (1.0 + null_stats.size)
    if alternative == "two-sided":
        return (1.0 + (np.abs(null_stats) >= abs(observed)).sum()) / (1.0 + null_stats.size)
    raise ValueError(f"unknown alternative: {alternative}")


def permutation_test(stat_fn: Callable, group_labels: np.ndarray, n_perm: int,
                     rng: np.random.Generator,
                     strata: Optional[np.ndarray] = None) -> Tuple[float, np.ndarray]:
    """Label-permutation test of ``stat_fn``.

    Args:
        stat_fn: callable taking ``(labels)`` and returning the statistic.
            The observed statistic is ``stat_fn(group_labels)``.
        group_labels: length-n binary treatment labels.
        n_perm: number of permutations.
        rng: numpy Generator.
        strata: optional length-n stratum ids; labels are permuted only within
            each stratum (covariate-preserving null).

    Returns:
        ``(observed_stat, null_stats)``.
    """
    labels = np.asarray(group_labels).copy()
    n = labels.size
    null_stats = np.empty(n_perm)
    observed = stat_fn(labels)

    if strata is None:
        for b in range(n_perm):
            null_stats[b] = stat_fn(labels[rng.permutation(n)])
    else:
        strata = np.asarray(strata)
        permuted = labels.copy()
        for s in np.unique(strata):
            idx = np.flatnonzero(strata == s)
            permuted[idx] = labels[idx][rng.permutation(idx.size)]
        for b in range(n_perm):
            for s in np.unique(strata):
                idx = np.flatnonzero(strata == s)
                permuted[idx] = labels[idx][rng.permutation(idx.size)]
            null_stats[b] = stat_fn(permuted)
    return float(observed), null_stats


def multiplier_bootstrap(influence_matrix: np.ndarray, n_draws: int,
                         rng: np.random.Generator,
                         kind: str = "gaussian") -> np.ndarray:
    """Multiplier bootstrap over a per-unit influence matrix.

    Args:
        influence_matrix: ``(n, resolution)`` per-unit influence-function
            values (row = unit). Convention matches tcda_uq's ``scores``.
        n_draws: number of null draws.
        rng: numpy Generator.
        kind: multiplier law: "gaussian" (N(0,1)) or "rademacher" (+-1).

    Returns:
        ``n_draws`` null statistics ``sup_t | n^{-1/2} sum_i g_i inf_i(t) |``.
        This is on the same scale as ``sqrt(n) * sup_t |mean_i inf_i(t)|``, the
        studentised statistic ``T_n`` of the plan, so observed values may be
        compared with these draws directly.
    """
    inf = np.asarray(influence_matrix, dtype=float)
    n = inf.shape[0]
    n_res = inf.shape[1]
    null_stats = np.empty(n_draws)
    scaled = inf / np.sqrt(n)
    for b in range(n_draws):
        if kind == "gaussian":
            g = rng.standard_normal(n)
        elif kind == "rademacher":
            g = rng.choice([-1.0, 1.0], size=n)
        else:
            raise ValueError(f"unknown multiplier kind: {kind}")
        curve = g @ scaled
        null_stats[b] = np.max(np.abs(curve))
    return null_stats


def paired_bootstrap(stat_fn: Callable, group_labels: np.ndarray, n_draws: int,
                     rng: np.random.Generator) -> Tuple[float, np.ndarray]:
    """Unit resampling with replacement preserving the treated/control split.

    Args:
        stat_fn: callable taking ``(labels)``.
        group_labels: length-n binary labels.
        n_draws: number of bootstrap draws.

    Returns:
        ``(observed_stat, bootstrap_stats)``.
    """
    labels = np.asarray(group_labels)
    n = labels.size
    n1 = int((labels == 1).sum())
    observed = stat_fn(labels)
    null_stats = np.empty(n_draws)
    for b in range(n_draws):
        idx = np.concatenate([
            rng.choice(np.flatnonzero(labels == 1), size=n1, replace=True),
            rng.choice(np.flatnonzero(labels == 0), size=n - n1, replace=True),
        ])
        null_stats[b] = stat_fn(labels[idx])
    return float(observed), null_stats


def smoothed_bootstrap(diagrams: Sequence[Sequence[np.ndarray]], n_draws: int,
                       rng: np.random.Generator, sigma: float,
                       stat_fn: Callable) -> Tuple[float, np.ndarray]:
    """Smoothed (jittered) bootstrap for persistent Betti numbers.

    Roycraft-Krebs-Polonik: the naive bootstrap is inconsistent for persistent
    Betti numbers; jittering (birth, death) coordinates by N(0, sigma^2) fixes
    the boundary effects.

    Args:
        diagrams: list over samples of list of (k, 2) per-dim diagrams.
        n_draws: number of bootstrap samples.
        rng: numpy Generator.
        sigma: jitter bandwidth (e.g. bandwidth / 2 of the kernel density).
        stat_fn: callable taking a list of diagram-lists (one per sample) and
            returning the statistic.

    Returns:
        ``(observed_stat, bootstrap_stats)``.
    """
    diagrams = [[np.asarray(d, dtype=float).reshape(-1, 2) for d in per_dim]
                for per_dim in diagrams]
    observed = stat_fn(diagrams)
    null_stats = np.empty(n_draws)
    n_samples = len(diagrams)
    for b in range(n_draws):
        # Resample WHOLE samples (diagram lists) with replacement -- one drawn
        # index per bootstrap unit. Pooling several units into a single diagram
        # would multiply every feature count by the pool size.
        idx = rng.integers(0, n_samples, size=n_samples)
        resampled = []
        for i in idx:
            per_dim = []
            for dgm in diagrams[i]:
                if dgm.size == 0:
                    per_dim.append(np.zeros((0, 2)))
                    continue
                jittered = dgm + rng.normal(0.0, sigma, size=dgm.shape)
                # keep the diagram above the diagonal after jittering
                jittered[:, 1] = np.maximum(jittered[:, 1], jittered[:, 0])
                per_dim.append(jittered)
            resampled.append(per_dim)
        null_stats[b] = stat_fn(resampled)
    return float(observed), null_stats


def cross_fit_folds(n: int, k_folds: int, rng: np.random.Generator,
                    stratify_labels: Optional[np.ndarray] = None) -> List[Tuple[np.ndarray, np.ndarray]]:
    """k-fold cross-fitting index splits.

    Returns:
        List of ``(train_idx, test_idx)`` pairs covering all n indices exactly
        once as test indices. If ``stratify_labels`` is given, class balance is
        preserved within folds.
    """
    if stratify_labels is None:
        perm = rng.permutation(n)
        fold_of = np.zeros(n, dtype=int)
        for f in range(k_folds):
            fold_of[perm[f::k_folds]] = f
    else:
        labels = np.asarray(stratify_labels)
        fold_of = np.empty(n, dtype=int)
        for lab in np.unique(labels):
            idx = np.flatnonzero(labels == lab)
            perm = rng.permutation(idx.size)
            for f in range(k_folds):
                fold_of[idx[perm[f::k_folds]]] = f
    folds = []
    for f in range(k_folds):
        test_idx = np.flatnonzero(fold_of == f)
        train_idx = np.flatnonzero(fold_of != f)
        folds.append((train_idx, test_idx))
    return folds

In [ ]:
%%writefile tda2s/resample/smoothing.py
"""Betti-curve helpers for the smoothed bootstrap (Roycraft-Krebs-Polonik).

Re-exports the canonical ``betti_curve`` from ``tda2s.vec`` and adds the
sample-level aggregate used by smoothed-bootstrap statistics.
"""
from __future__ import annotations

import numpy as np

from tda2s.vec import betti_curve as _vec_betti_curve


def _max_death(diagrams):
    """Largest finite death across a list of per-dim (k, 2) arrays."""
    m = 0.0
    for dgm in diagrams:
        dgm = np.asarray(dgm, dtype=float)
        if dgm.ndim == 2 and dgm.size:
            finite = dgm[np.isfinite(dgm[:, 1]), 1]
            if finite.size:
                m = max(m, float(finite.max()))
    return m


def betti_curve(diagrams, interval=None, n_points=100):
    """Persistent Betti-number curve of ONE sample's diagram list.

    Args:
        diagrams: list of (k, 2) per-dim (birth, death) arrays (one sample).
        interval: (t_min, t_max) grid; defaults to (0, max death).
        n_points: grid resolution.

    Returns:
        ``(t_grid, betti_matrix)`` with ``betti_matrix[d]`` the B_d(t) curve.
    """
    iv = interval if interval is not None else (0.0, _max_death(diagrams) + 1e-9)
    grid = np.linspace(iv[0], iv[1], n_points)
    matrix = _vec_betti_curve(diagrams, interval=iv, n_points=n_points)
    return grid, matrix


def mean_betti_curve(sample_diagrams, interval=None, n_points=100):
    """Mean Betti curve over a sample of diagrams (for bootstrap statistics).

    All samples are evaluated on ONE shared grid. When ``interval`` is not
    given it is derived from the *pooled* diagrams, not per sample: averaging
    curves that were each sampled on their own grid would mix incomparable
    abscissae and silently distort the mean.

    Args:
        sample_diagrams: list over samples of list of (k, 2) per-dim arrays.
        interval: shared (t_min, t_max); defaults to (0, pooled max death).
        n_points: grid resolution.

    Returns:
        ``(t_grid, mean_betti)`` with ``mean_betti[d]`` a length-``n_points``
        curve averaged over the sample.
    """
    if interval is None:
        pooled = 0.0
        for diags in sample_diagrams:
            pooled = max(pooled, _max_death(diags))
        interval = (0.0, pooled + 1e-9) if pooled > 0 else (0.0, 1.0)
    grid = np.linspace(interval[0], interval[1], n_points)
    if not sample_diagrams:
        return grid, np.zeros((1, n_points))
    curves = [_vec_betti_curve(diags, interval=interval, n_points=n_points)
              for diags in sample_diagrams]
    return grid, np.mean(np.stack(curves), axis=0)

In [ ]:
%%writefile tda2s/benchmarks/__init__.py
"""Competitor two-sample tests on persistence diagrams (Phase 0.5).

Every wrapper takes ``(diags0, diags1)`` first and returns a p-value in
[0, 1], where ``diags0``/``diags1`` are lists over samples of lists (per
homology dim) of ``(k, 2)`` birth-death arrays.

Their *keyword* arguments are NOT uniform: six of the seven are permutation
tests taking ``n_perm``/``seed``, but ``moon_lazar`` is analytic (pooled-variance
t-tests + Benjamini-Hochberg) and accepts neither. Call through
:func:`run_competitor` to sweep the registry with one kwargs dict; it drops the
arguments a given wrapper does not accept instead of raising ``TypeError``.

None of these methods ship author code, so every wrapper is a transcription of
its source paper with the section and equation numbers cited in the module
docstring. ``tests/test_published_reproductions.py`` reproduces a published
figure for each method that has one.

Competitors:
    * ``rt``              - Robinson & Turner (2017) permutation test on
                            pairwise Wasserstein distances.
    * ``mmd``             - Kwitt et al. (2015) kernel MMD on diagram points.
    * ``han``             - Han, Kim & Kim (2026) kernel permutation test on
                            weighted persistence intensity functions.
    * ``strand``          - Murris, Stolz & Borgwardt (2026) log-rank test on
                            feature lifetimes, stratified by homology dim.
    * ``moon_lazar``      - Moon & Lazar (2023) Algorithm 1: persistence images,
                            variance pre-filter, pooled-variance t-tests, FDR.
    * ``frechet_anova``   - Dubey & Muller (2019) Frechet ANOVA, eqs. (6)-(11),
                            including the Levene-type ``U_n`` term.
    * ``krebs_rademacher``- Krebs & Rademacher (2024) Section 1.2 relevant-
                            difference test on Wasserstein inco-variances.
                            Targets DISPERSION, not location: it is blind to a
                            pure location shift by construction (eq. 1.7).
"""
from .frechet_anova import test_frechet_anova
from .han import han_kernels, test_han, test_han_from_kernels
from .krebs_rademacher import test_krebs_rademacher
from .mmd import mmd_gram, test_mmd, test_mmd_from_gram
from .moon_lazar import test_moon_lazar
from .rt import test_rt, test_rt_from_matrix
from .strand import test_strand

COMPETITORS = {
    "rt": test_rt,
    "mmd": test_mmd,
    "han": test_han,
    "strand": test_strand,
    "moon_lazar": test_moon_lazar,
    "frechet_anova": test_frechet_anova,
    "krebs_rademacher": test_krebs_rademacher,
}

#: Wrappers that are analytic rather than permutation-based.
ANALYTIC = frozenset({"moon_lazar"})


def run_competitor(name, diags0, diags1, **kwargs):
    """Run competitor ``name``, passing only the kwargs it actually accepts.

    Lets a caller sweep the whole registry with a single kwargs dict (e.g.
    ``n_perm=200, seed=0``) even though the analytic wrappers take no such
    arguments.

    Args:
        name: key of :data:`COMPETITORS`.
        diags0, diags1: the two groups of diagram lists.
        **kwargs: candidate keyword arguments; unsupported ones are dropped.

    Returns:
        float p-value in [0, 1].
    """
    import inspect

    try:
        fn = COMPETITORS[name]
    except KeyError:
        raise ValueError(
            f"unknown competitor {name!r}; expected one of {sorted(COMPETITORS)}") from None
    accepted = inspect.signature(fn).parameters
    return float(fn(diags0, diags1,
                    **{k: v for k, v in kwargs.items() if k in accepted}))


__all__ = ["COMPETITORS", "ANALYTIC", "run_competitor", "test_rt", "test_mmd",
           "test_han", "test_strand", "test_moon_lazar", "test_frechet_anova",
           "test_krebs_rademacher",
           # label-independent precompute / read-back pairs, for sweeps that
           # share one pooled sample across many group splits (Phase 2).
           "test_rt_from_matrix", "mmd_gram", "test_mmd_from_gram",
           "han_kernels", "test_han_from_kernels"]


In [ ]:
%%writefile tda2s/benchmarks/_common.py
"""Shared internal helpers for the competitor wrappers in ``tda2s.benchmarks``.

Implementation details of the wrappers, not part of the public API.

Diagram conventions (uniform across wrappers):
    * ``diags0`` / ``diags1``: lists over samples of lists (indexed by homology
      dim) of ``(k, 2)`` birth-death arrays.
    * All samples of a group share the same number of homology dimensions.
"""
from __future__ import annotations

import numpy as np


def _validate(diags0, diags1):
    """Validate the uniform diagram-list input format.

    Returns:
        ``(diags0, diags1, n_dims)`` with the diagrams as plain lists and the
        (common) number of homology dimensions.
    """
    d0 = [list(d) for d in diags0]
    d1 = [list(d) for d in diags1]
    if not d0 or not d1:
        raise ValueError("each group must contain at least one diagram")
    nd = len(d0[0])
    for d in d0 + d1:
        if len(d) != nd:
            raise ValueError(
                "all diagrams must contain the same number of homology dims")
    return d0, d1, nd


def _points(dgm):
    """``(k, 2)`` birth-death array -> float array, dropping empty/infinite rows."""
    a = np.asarray(dgm, dtype=float)
    if a.ndim != 2 or a.size == 0:
        return np.empty((0, 2))
    return a[np.isfinite(a).all(axis=1)]


def _persistence(dgm):
    a = _points(dgm)
    return a[:, 1] - a[:, 0]


def _flatten_points(diags):
    """All points (across samples and homology dims) -> ``(n, 2)`` array."""
    blocks = [_points(dgm) for d in diags for dgm in d]
    blocks = [b for b in blocks if len(b)]
    return np.vstack(blocks) if blocks else np.empty((0, 2))


def _median_pairwise_scale(points, coords=None, max_sample=2000, seed=0):
    """Median pairwise absolute difference along each coordinate of ``points``.

    Used to set per-coordinate bandwidths for kernel-based tests.  The
    subsample is deterministic (fixed internal seed) so results do not depend
    on the calling test's RNG.
    """
    pts = np.asarray(points, dtype=float)
    if len(pts) < 2:
        return np.ones(max(1, pts.shape[1] if pts.ndim > 1 else 1))
    rng = np.random.default_rng(seed)
    if len(pts) > max_sample:
        pts = pts[rng.choice(len(pts), max_sample, replace=False)]
    if coords is None:
        coords = range(pts.shape[1])
    scales = []
    for j in coords:
        d = np.abs(pts[:, j, None] - pts[None, :, j])
        d = d[np.triu_indices(len(pts), 1)]
        s = np.median(d)
        scales.append(float(s) if np.isfinite(s) and s > 0 else 1.0)
    return np.asarray(scales)


def _median_euclidean(points, max_sample=2000, seed=0):
    """Median pairwise Euclidean distance (median-heuristic bandwidth)."""
    pts = np.asarray(points, dtype=float)
    if len(pts) < 2:
        return 1.0
    rng = np.random.default_rng(seed)
    if len(pts) > max_sample:
        pts = pts[rng.choice(len(pts), max_sample, replace=False)]
    d = np.linalg.norm(pts[:, None, :] - pts[None, :, :], axis=2)
    d = d[np.triu_indices(len(pts), 1)]
    med = np.median(d)
    return float(med) if np.isfinite(med) and med > 0 else 1.0


#: Column-block width for :func:`_gaussian_gram_blocks`. ``n_points x CHUNK``
#: float64 is the peak allocation, so 1024 costs ~150 MB at 18k pooled points.
_GRAM_CHUNK = 1024


def _gaussian_gram_blocks(X, B, gamma=1.0, chunk=_GRAM_CHUNK):
    """``B @ exp(-gamma * ||x_i - x_j||^2) @ B.T`` without forming the full Gram.

    The kernel two-sample tests need only the ``(n_samples, n_samples)`` matrix
    of diagram-level kernel sums, but the naive route builds the point-level
    ``(n_points, n_points)`` Gram first. At the Phase 0 benchmark size (100
    clouds, H0 + H1, ~18k pooled points) that intermediate is 2.6 GB, and
    writing it as ``X[:, None, :] - X[None, :, :]`` costs a further 7.8 GB
    because the difference is materialised in 3-D before being reduced. This
    accumulates the same product one column block at a time instead, so peak
    memory is ``O(n_points * chunk)`` and the result is identical up to
    floating-point summation order.

    Args:
        X: ``(n_points, d)`` point coordinates.
        B: ``(n_samples, n_points)`` membership matrix, float. Per-point
            weights are folded in by scaling its columns, since a weighted
            kernel ``w(x) k(x, y) w(y)`` equals ``(B w) k (B w)^T``.
        gamma: exponent scale; pass ``1 / (2 sigma^2)`` for bandwidth ``sigma``.
        chunk: columns of the Gram per block.

    Returns:
        ``(n_samples, n_samples)`` array.
    """
    from scipy.spatial.distance import cdist

    npts = X.shape[0]
    S = np.empty((B.shape[0], npts))
    for start in range(0, npts, chunk):
        stop = min(start + chunk, npts)
        D2 = cdist(X, X[start:stop], "sqeuclidean")
        S[:, start:stop] = B @ np.exp(-gamma * D2)
    return S @ B.T


def _membership(counts, n_samples, weights=None):
    """``(n_samples, n_points)`` block-membership matrix for :func:`_gaussian_gram_blocks`.

    ``counts[i]`` points belong to sample ``i``, laid out contiguously in the
    order the samples were pooled. ``weights`` (per point) is folded into the
    columns when given.
    """
    npts = int(np.sum(counts))
    B = np.zeros((n_samples, npts))
    off = 0
    for i, c in enumerate(counts):
        B[i, off:off + c] = 1.0
        off += c
    if weights is not None:
        B *= np.asarray(weights, dtype=float)[None, :]
    return B


def _perm_groups(N, n0, n_perm, rng):
    """``(n_perm, N)`` boolean label arrays, exactly ``n0`` True entries per row."""
    return np.stack([rng.permutation(N) < n0 for _ in range(n_perm)])


def _perm_pvalue(obs, null, direction="greater"):
    """Phipson-Smyth permutation p-value: ``(1 + #extreme) / (1 + n_perm)``."""
    null = np.asarray(null, dtype=float)
    if direction == "greater":
        cnt = np.count_nonzero(null >= obs)
    else:
        cnt = np.count_nonzero(null <= obs)
    return float((1.0 + cnt) / (1.0 + len(null)))


def _block_means(P, g):
    """Group-split means of an ``N x N`` diagram-pair matrix ``P``.

    Args:
        P: symmetric matrix with ``P[i, j]`` = cost/kernel between diagrams i, j.
        g: ``(N,)`` bool labels (True = group 0).

    Returns:
        ``(within0, within1, between)``: mean off-diagonal ``P`` over pairs
        within group 0, within group 1, and across the two groups.
    """
    g = np.asarray(g, dtype=bool)
    i0, i1 = np.flatnonzero(g), np.flatnonzero(~g)
    n0, n1 = len(i0), len(i1)
    w0 = (P[np.ix_(i0, i0)].sum() - np.diag(P)[i0].sum()) / (n0 * (n0 - 1)) if n0 > 1 else 0.0
    w1 = (P[np.ix_(i1, i1)].sum() - np.diag(P)[i1].sum()) / (n1 * (n1 - 1)) if n1 > 1 else 0.0
    b = P[np.ix_(i0, i1)].sum() / (n0 * n1) if n0 and n1 else 0.0
    return float(w0), float(w1), float(b)


def _persistence_vectors(diags0, diags1, max_len=None):
    """Sorted (descending) zero-padded persistence vectors per diagram.

    Homology dims are concatenated; within each dim every diagram is padded
    with zeros to a common length (the max feature count across both groups,
    optionally capped at ``max_len``).

    Returns:
        ``(V0, V1)`` with shapes ``(n0, L)`` and ``(n1, L)``.
    """
    d0, d1, nd = _validate(diags0, diags1)
    parts0, parts1 = [], []
    for dim in range(nd):
        p0 = [_persistence(d[dim]) for d in d0]
        p1 = [_persistence(d[dim]) for d in d1]
        L = max([len(p) for p in p0 + p1] or [0])
        if max_len is not None:
            L = min(L, max_len)
        if L == 0:
            parts0.append(np.zeros((len(d0), 0)))
            parts1.append(np.zeros((len(d1), 0)))
            continue
        v0 = np.zeros((len(d0), L))
        v1 = np.zeros((len(d1), L))
        for i, p in enumerate(p0):
            p = np.sort(p)[::-1][:L]
            v0[i, :len(p)] = p
        for i, p in enumerate(p1):
            p = np.sort(p)[::-1][:L]
            v1[i, :len(p)] = p
        parts0.append(v0)
        parts1.append(v1)
    return (np.hstack(parts0) if parts0 else np.zeros((len(d0), 0)),
            np.hstack(parts1) if parts1 else np.zeros((len(d1), 0)))


In [ ]:
%%writefile tda2s/benchmarks/rt.py
"""Robinson & Turner (2017) permutation two-sample test for persistence diagrams.

Reference: A. Robinson and K. Turner, "Hypothesis testing for topological data
analysis", Journal of Applied and Computational Topology 1 (2017) 241-261
(arXiv:1310.7467).

The test is a randomization (label-permutation) test on a "joint loss
function".  For each homology dimension ``d`` a distance ``d_p`` between two
diagrams is fixed (bottleneck distance, p = infinity, by default; Wasserstein
W_1 via ``persim`` for ``metric="wasserstein"``).  The test statistic is the
sum over homology dimensions of the mean pairwise within-group distance over
the two groups (the paper's ``F_{p, q}`` family with q = 1):

    F(L) = sum_dims [ mean_{i<j, both in group 0} d_p(D_i, D_j)
                   + mean_{i<j, both in group 1} d_p(D_i, D_j) ].

If the grouping is sensible (alternative), within-group distances are small,
so we reject for *small* observed F: the p-value is the proportion of random
labelings whose loss is at most the observed loss, in the Phipson-Smyth
``(1 + count) / (1 + n_perm)`` convention used in the paper's Algorithm 2.

The pairwise distance matrix is precomputed once and read back under each
permutation, so the permutation loop is cheap.

``statistic="within"`` (default) reproduces the paper's joint loss.  The
``statistic="between"`` variant instead uses the mean pairwise distance
between the two groups' diagrams and rejects for large values; it is the
version described in the Phase 0.5 benchmark spec.
"""
from __future__ import annotations

import numpy as np

from ._common import _block_means, _perm_groups, _perm_pvalue, _points, _validate


def _diagram_distance(dgm1, dgm2, metric):
    a, b = _points(dgm1), _points(dgm2)
    if metric == "bottleneck":
        import gudhi as gd
        return gd.bottleneck_distance(a, b)
    if metric == "wasserstein":
        import persim
        return persim.wasserstein(a, b)
    raise ValueError(f"unknown metric: {metric!r}")


def test_rt_from_matrix(P, group, n_perm=200, statistic="within", seed=None,
                        observed=None):
    """Robinson-Turner p-value from a precomputed pairwise distance matrix.

    The pairwise matrix of a pooled sample does not depend on the labels, so
    sweeps that reuse one set of diagrams across many group assignments (e.g.
    the Phase 2 imbalance sweep, where the clouds are shared across propensity
    strengths) must compute ``P`` once and read it back here per split. This
    also keeps the expensive ``O(N^2)`` bottleneck loop out of the permutation
    count.

    Args:
        P: symmetric ``N x N`` matrix, ``P[i, j]`` = joint loss contribution
            of diagrams i, j (summed over homology dims by the caller).
        group: the observed labelling, **in the row order of** ``P``. Either an
            ``(N,)`` boolean mask with ``True`` = group 0, or an integer ``n0``
            meaning "the first ``n0`` rows are group 0". The integer form is
            only correct when ``P``'s rows are already sorted by label; a
            caller that shares one matrix across several label draws (the Phase
            2 sweep) must pass the mask, or the observed statistic is evaluated
            under an arbitrary labelling and the p-value degenerates to a draw
            from the null.
        n_perm: number of label permutations (default 200).
        statistic: ``"within"`` (joint loss; reject for small) or
            ``"between"`` (mean cross-group distance; reject for large).
        seed: RNG seed for the permutations.
        observed: optional precomputed observed statistic; computed when None.

    Returns:
        float p-value in [0, 1].
    """
    N = P.shape[0]
    rng = np.random.default_rng(seed)
    if np.ndim(group) == 0:
        n0 = int(group)
        obs_group = np.zeros(N, dtype=bool)
        obs_group[:n0] = True
    else:
        obs_group = np.asarray(group, dtype=bool)
        if obs_group.shape != (N,):
            raise ValueError(
                f"group mask has shape {obs_group.shape}, expected ({N},)")
        n0 = int(obs_group.sum())
    if observed is None:
        w0, w1, b = _block_means(P, obs_group)
        observed = (w0 + w1) if statistic == "within" else b
    perms = _perm_groups(N, n0, n_perm, rng)
    null = np.empty(n_perm)
    for k in range(n_perm):
        w0, w1, b = _block_means(P, perms[k])
        null[k] = (w0 + w1) if statistic == "within" else b
    direction = "less" if statistic == "within" else "greater"
    return _perm_pvalue(observed, null, direction)


def test_rt(diags0, diags1, metric="bottleneck", statistic="within",
            n_perm=200, seed=None):
    """Robinson-Turner permutation test between two groups of diagrams.

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays.
        metric: pairwise diagram distance, "bottleneck" (default) or
            "wasserstein" (W_1).
        statistic: "within" (paper's joint loss; reject for small values) or
            "between" (mean cross-group distance; reject for large values).
        n_perm: number of label permutations (default 200).
        seed: RNG seed for the permutations.

    Returns:
        float p-value in [0, 1].
    """
    d0, d1, nd = _validate(diags0, diags1)
    n0, n1 = len(d0), len(d1)
    N = n0 + n1
    pooled = d0 + d1
    rng = np.random.default_rng(seed)

    P = np.zeros((N, N))
    for dim in range(nd):
        for i in range(N):
            for j in range(i + 1, N):
                v = _diagram_distance(pooled[i][dim], pooled[j][dim], metric)
                P[i, j] = P[j, i] = v

    obs_group = np.zeros(N, dtype=bool)
    obs_group[:n0] = True
    w0, w1, b = _block_means(P, obs_group)
    obs = (w0 + w1) if statistic == "within" else b

    perms = _perm_groups(N, n0, n_perm, rng)
    null = np.empty(n_perm)
    for k in range(n_perm):
        w0, w1, b = _block_means(P, perms[k])
        null[k] = (w0 + w1) if statistic == "within" else b

    direction = "less" if statistic == "within" else "greater"
    return _perm_pvalue(obs, null, direction)


In [ ]:
%%writefile tda2s/benchmarks/mmd.py
"""Kernel maximum mean discrepancy (MMD) two-sample test on diagram points.

Reference: R. Kwitt, S. Huber, M. Niethammer, W. Lin and U. Bauer,
"Statistical topological data analysis - a kernel perspective", NeurIPS 2015.

Each diagram is embedded as the (unweighted) sum of Gaussian kernels over its
points in ``(b, d, persistence)`` (default) or ``(b, d)`` coordinates:

    k(p, q) = exp(-||p - q||^2 / (2 sigma^2)),
    K(D, D') = sum_{p in D} sum_{q in D'} k(p, q).

The two groups are compared through the biased MMD^2 estimate between the
empirical kernel-mean embeddings of the two diagram distributions; sigma is
set by the median heuristic over the pooled point set.  The null distribution
is obtained by permuting the diagram labels (200 permutations by default).

The diagram-level kernel matrix is precomputed once as block sums of the
point-level Gaussian kernel, so the permutation loop is cheap. Those block sums
are accumulated one column block at a time (``_gaussian_gram_blocks``): the
point-level Gram is ~18k x 18k at Phase 0 benchmark size, which is 2.6 GB that
the ``(n_samples, n_samples)`` result never requires.
"""
from __future__ import annotations

import numpy as np

from ._common import (_gaussian_gram_blocks, _median_euclidean, _membership,
                      _perm_groups, _perm_pvalue, _points, _validate)


def _point_features(diags, coords, epsilon=0.0):
    """Per-sample coordinate matrix for all diagram points of ``diags``."""
    feats = []
    counts = []
    for d in diags:
        block = []
        for dgm in d:
            p = _points(dgm)
            if epsilon > 0.0 and len(p):
                p = p[p[:, 1] - p[:, 0] >= epsilon]
            if len(p):
                if coords == "bdp":
                    p = np.column_stack([p, p[:, 1] - p[:, 0]])
                block.append(p)
        x = np.vstack(block) if block else np.empty((0, 2 if coords == "bd" else 3))
        feats.append(x)
        counts.append(len(x))
    return feats, counts


def _mmd2(P, g):
    """Biased MMD^2 between the two groups given labels ``g`` (bool array)."""
    g = np.asarray(g, dtype=bool)
    n0, n1 = int(g.sum()), len(g) - int(g.sum())
    w0 = P[np.ix_(g, g)].sum() / (n0 * n0)
    w1 = P[np.ix_(~g, ~g)].sum() / (n1 * n1)
    b = P[np.ix_(g, ~g)].sum() / (n0 * n1)
    return float(w0 + w1 - 2.0 * b)


def mmd_gram(diags, sigma=None, coords="bdp", epsilon=0.0):
    """Diagram-level kernel matrix of a pooled sample, before any labelling.

    The matrix, and the median-heuristic bandwidth that scales it, depend only
    on the pooled diagrams, so a caller comparing several label draws over one
    sample (the Phase 2 imbalance sweep) builds it once here and reads it back
    per split through :func:`test_mmd_from_gram`.

    Args:
        diags: list over samples of lists (per homology dim) of ``(k, 2)``
            birth-death arrays, in whatever order the caller will label them.
        sigma: Gaussian bandwidth; ``None`` (default) = median heuristic.
        coords: "bdp" (birth, death, persistence) or "bd".
        epsilon: near-diagonal persistence filter (see :func:`test_mmd`).

    Returns:
        ``(P, sigma)`` with ``P`` the ``(N, N)`` kernel matrix, or
        ``(None, None)`` when the pooled sample has no finite diagram points.
    """
    pooled = [list(d) for d in diags]
    N = len(pooled)
    feats, counts = _point_features(pooled, coords, epsilon=epsilon)
    if sum(counts) == 0:
        return None, None
    X = np.vstack(feats)
    if sigma is None:
        sigma = _median_euclidean(X)
    P = _gaussian_gram_blocks(X, _membership(counts, N),
                              gamma=1.0 / (2.0 * sigma * sigma))
    return P, float(sigma)


def test_mmd_from_gram(P, group, n_perm=200, seed=None):
    """MMD p-value from a precomputed diagram-level kernel matrix.

    Args:
        P: ``(N, N)`` matrix from :func:`mmd_gram`, or ``None`` (degenerate
            sample; the p-value is then 1).
        group: ``(N,)`` boolean mask in ``P``'s row order, ``True`` = group 0.
        n_perm: number of label permutations (default 200).
        seed: RNG seed for the permutations.

    Returns:
        float p-value in [0, 1].
    """
    if P is None:
        return 1.0
    g = np.asarray(group, dtype=bool)
    N = P.shape[0]
    if g.shape != (N,):
        raise ValueError(f"group mask has shape {g.shape}, expected ({N},)")
    n0 = int(g.sum())
    if n0 == 0 or n0 == N:
        raise ValueError("each group must contain at least one diagram")
    rng = np.random.default_rng(seed)
    obs = _mmd2(P, g)
    perms = _perm_groups(N, n0, n_perm, rng)
    null = np.array([_mmd2(P, perms[k]) for k in range(n_perm)])
    return _perm_pvalue(obs, null, "greater")


def test_mmd(diags0, diags1, sigma=None, coords="bdp", n_perm=200, seed=None,
             epsilon=0.0):
    """Kernel MMD two-sample test between two groups of diagrams.

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays.
        sigma: Gaussian bandwidth; ``None`` (default) = median heuristic over
            the pooled point set.
        coords: point coordinates, "bdp" = (birth, death, persistence) or
            "bd" = (birth, death).
        n_perm: number of label permutations (default 200).
        seed: RNG seed for the permutations.
        epsilon: diagram points with persistence below this threshold are
            dropped before the kernel is built (default 0.0 = the published
            embedding over all points).  By the boundedness of the Gaussian
            kernel each dropped point perturbs the diagram-level kernel by a
            tiny amount; the permutation null remains exactly valid for any
            embedding, so ``epsilon`` only trades a negligible power shift
            against the O(n^2) point-level Gram cost (a sweep convenience).

    Returns:
        float p-value in [0, 1].
    """
    d0, d1, _ = _validate(diags0, diags1)
    n0, n1 = len(d0), len(d1)
    N = n0 + n1

    # diagram-level kernel matrix, accumulated blockwise: the point-level Gram
    # is ~18k x 18k at benchmark size and is never needed in full.
    P, _ = mmd_gram(d0 + d1, sigma=sigma, coords=coords, epsilon=epsilon)

    obs_group = np.zeros(N, dtype=bool)
    obs_group[:n0] = True
    return test_mmd_from_gram(P, obs_group, n_perm=n_perm, seed=seed)


In [ ]:
%%writefile tda2s/benchmarks/han.py
"""Han, Kim & Kim (2026) kernel permutation test on persistence intensity functions.

Reference: Y. Han, I. Kim and J. Kim, "A two-sample test on weighted
persistence intensity functions in topological data analysis", arXiv:2607.20893
(2026).  Paper accessed in full on arXiv; this is the method as described
there (not the intensity-grid/sup-norm/bootstrapped version sketched in the
Phase 0.5 brief, which does not match the published procedure).

Method (paper Sections 2-4 and 6):
    * Diagram points z = (b, d) in Omega = {y > x >= 0} carry a weight
      ``w(z) = (d - b)^q`` (the paper also allows arctan; q = weight_power,
      default 1).
    * The kernel is a product of two 1D kernels with a bandwidth vector
      ``lambda = (lambda_1, lambda_2)``,
      ``k_lambda(x, y) = prod_i (1/(lambda_i sqrt(pi))) exp(-((x_i - y_i)/lambda_i)^2)``,
      and the weighted diagram kernel is
      ``K(X, Y) = sum_{x in X} sum_{y in Y} w(x) w(y) k_lambda(x, y)``.
    * The test statistic is the unbiased two-sample U-statistic estimator of
      ``||mu_p - mu_q||^2`` in the RKHS of ``k_{w, lambda}``:
      ``T = E_{i != i'} K(U_i, U_i') + E_{j != j'} K(U_j, U_j') - 2 E_{i, j} K(U_i, U_j)``.
    * The null distribution is the permutation distribution of T (the paper's
      Algorithm 1).  Because the optimal bandwidth is unknown, the default
      ``aggregate=True`` runs the bandwidth-aggregation test (Aggtest,
      Algorithm 2): per-bandwidth rank p-values are computed on the same
      permutations and combined as ``A_b = min_lambda p_b^lambda``; the final
      p-value counts how many of the B + 1 replicates (permutations plus the
      observation) attain ``A_b <= A_obs``.

The bandwidth grid is ``scale * m`` over ``scales``, where ``m`` is the
median pairwise absolute difference of the pooled diagram points along each
coordinate (a data-driven stand-in; the paper's simulation grid is reported
only in its supplementary material, which we could not fully retrieve).
"""
from __future__ import annotations

import numpy as np

from ._common import (_gaussian_gram_blocks, _median_pairwise_scale,
                      _membership, _perm_groups, _perm_pvalue, _points,
                      _validate)


def _weighted_diagram_kernel(X, Bw, lam):
    """Diagram-level weighted kernel matrix ``K(U_i, U_j)`` for bandwidth ``lam``.

    The paper's kernel is a product of one-dimensional Gaussians,
    ``prod_j (lam_j sqrt(pi))^-1 exp(-((x_j - y_j) / lam_j)^2)``, which is the
    isotropic ``exp(-||z - z'||^2)`` on rescaled coordinates ``z = x / lam``
    times the constant ``prod_j (lam_j sqrt(pi))^-1``. The point weights
    ``w(x) w(y)`` are already folded into the columns of ``Bw``, so the whole
    diagram-level matrix is one blockwise Gram accumulation and the point-level
    kernel is never materialised.
    """
    lam = np.asarray(lam, dtype=float)
    norm = 1.0 / np.prod(lam * np.sqrt(np.pi))
    return norm * _gaussian_gram_blocks(X / lam[None, :], Bw, gamma=1.0)


def _u_statistic(P, g):
    """Unbiased two-sample U-statistic on the diagram kernel matrix ``P``."""
    g = np.asarray(g, dtype=bool)
    i0, i1 = np.flatnonzero(g), np.flatnonzero(~g)
    n0, n1 = len(i0), len(i1)
    w0 = (P[np.ix_(i0, i0)].sum() - np.diag(P)[i0].sum()) / (n0 * (n0 - 1)) if n0 > 1 else 0.0
    w1 = (P[np.ix_(i1, i1)].sum() - np.diag(P)[i1].sum()) / (n1 * (n1 - 1)) if n1 > 1 else 0.0
    b = P[np.ix_(i0, i1)].sum() / (n0 * n1) if n0 and n1 else 0.0
    return float(w0 + w1 - 2.0 * b)


def han_kernels(diags, weight="persistence", weight_power=1.0,
                scales=(0.5, 1.0, 2.0, 4.0), aggregate=True, epsilon=0.0):
    """Per-bandwidth diagram kernel matrices of a pooled sample, before labelling.

    Both the bandwidth grid (a multiple of the pooled median pairwise scale)
    and the kernel matrices depend only on the pooled diagrams, so a caller
    comparing several label draws over one sample (the Phase 2 imbalance
    sweep) builds them once here and reads them back per split through
    :func:`test_han_from_kernels`. This is the whole cost of the test: the
    Aggtest permutation loop is block sums on the matrices returned here.

    Args:
        diags: list over samples of lists (per homology dim) of ``(k, 2)``
            birth-death arrays, in whatever order the caller will label them.
        weight, weight_power, scales, aggregate, epsilon: as in
            :func:`test_han`. ``aggregate`` decides the grid size only; pass
            the same value to :func:`test_han_from_kernels`.

    Returns:
        list of ``(N, N)`` matrices, one per bandwidth, or ``None`` when the
        pooled sample has no diagram points above ``epsilon``.
    """
    pooled = [list(d) for d in diags]
    N = len(pooled)
    X, counts = _flatten_bd(pooled, epsilon=epsilon)
    if len(X) == 0:
        return None
    pers = X[:, 1] - X[:, 0]
    if weight == "persistence":
        weights = pers ** float(weight_power)
    elif weight is None:
        weights = np.ones(len(X))
    else:
        raise ValueError(f"unknown weight: {weight!r}")

    Bw = _membership(counts, N, weights=weights)
    med = _median_pairwise_scale(X)
    grid = [med * s for s in scales] if aggregate else [med]
    return [_weighted_diagram_kernel(X, Bw, lam) for lam in grid]


def test_han_from_kernels(Ps, group, aggregate=True, n_perm=200, seed=None):
    """Han-Kim-Kim p-value from precomputed per-bandwidth kernel matrices.

    Args:
        Ps: list of ``(N, N)`` matrices from :func:`han_kernels`, or ``None``
            (degenerate sample; the p-value is then 1).
        group: ``(N,)`` boolean mask in the matrices' row order, ``True`` =
            group 0.
        aggregate: must match the value passed to :func:`han_kernels`.
        n_perm: number of label permutations (default 200).
        seed: RNG seed for the permutations.

    Returns:
        float p-value in [0, 1].
    """
    if Ps is None:
        return 1.0
    g = np.asarray(group, dtype=bool)
    N = Ps[0].shape[0]
    if g.shape != (N,):
        raise ValueError(f"group mask has shape {g.shape}, expected ({N},)")
    n0 = int(g.sum())
    if n0 == 0 or n0 == N:
        raise ValueError("each group must contain at least one diagram")
    rng = np.random.default_rng(seed)

    perms = _perm_groups(N, n0, n_perm, rng)
    stat_obs = np.empty(len(Ps))
    stat_null = np.zeros((len(Ps), n_perm))
    for li, P in enumerate(Ps):
        stat_obs[li] = _u_statistic(P, g)
        for k in range(n_perm):
            stat_null[li, k] = _u_statistic(P, perms[k])

    if not aggregate:
        return _perm_pvalue(stat_obs[0], stat_null[0], "greater")

    # Aggtest: rank p-values per bandwidth over the B + 1 replicates
    all_stats = np.concatenate([stat_null, stat_obs[:, None]], axis=1)  # (n_bw, B+1)
    B1 = n_perm + 1
    ranks = (B1 - np.argsort(np.argsort(all_stats, axis=1), axis=1)) / B1
    A = ranks.min(axis=0)
    return float((1.0 + np.count_nonzero(A <= A[-1])) / B1)


def test_han(diags0, diags1, weight="persistence", weight_power=1.0,
             scales=(0.5, 1.0, 2.0, 4.0), aggregate=True, n_perm=200,
             seed=None, epsilon=0.0):
    """Han-Kim-Kim kernel permutation test on weighted persistence intensity.

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays.
        weight: "persistence" (w(z) = (d - b)^weight_power, default) or None
            (unweighted).
        weight_power: exponent q of the persistence weight.
        scales: multipliers for the per-coordinate median pairwise scale that
            form the bandwidth grid (Aggtest).
        aggregate: if True (default), run the bandwidth-aggregated test
            (Aggtest); otherwise a single bandwidth ``(median, median)``.
        n_perm: number of label permutations (default 200).
        seed: RNG seed for the permutations.
        epsilon: diagram points with persistence below this threshold are
            dropped before the kernel is built (default 0.0 = the published
            weighted intensity over all points).  Near-diagonal points carry
            weight ``(d - b)^q < epsilon^q``, so the perturbation of the
            diagram kernel is bounded by their kernel contribution and the
            permutation null remains exactly valid; ``epsilon`` only trades a
            negligible power shift against the O(n^2) point-level Gram cost
            (a sweep convenience).

    Returns:
        float p-value in [0, 1].
    """
    d0, d1, _ = _validate(diags0, diags1)
    n0, n1 = len(d0), len(d1)
    N = n0 + n1

    Ps = han_kernels(d0 + d1, weight=weight, weight_power=weight_power,
                     scales=scales, aggregate=aggregate, epsilon=epsilon)
    g0 = np.zeros(N, dtype=bool)
    g0[:n0] = True
    return test_han_from_kernels(Ps, g0, aggregate=aggregate, n_perm=n_perm,
                                 seed=seed)


def _flatten_bd(diags, epsilon=0.0):
    pts, counts = [], []
    for d in diags:
        n = 0
        for dgm in d:
            p = _points(dgm)
            if epsilon > 0.0 and len(p):
                p = p[p[:, 1] - p[:, 0] >= epsilon]
            if len(p):
                pts.append(p)
                n += len(p)
        counts.append(n)
    return (np.vstack(pts) if pts else np.empty((0, 2))), counts

In [ ]:
%%writefile tda2s/benchmarks/strand.py
"""STRAND: survival-framing two-sample test for collections of persistence
diagrams (Murris, Stolz & Borgwardt 2026).

Reference: J. Murris, B. Stolz and K. Borgwardt, "From persistence to
survival: hypothesis testing, effect sizes and vectorisation for topological
features", arXiv:2606.11911 (2026).  Paper accessed in full on arXiv.

Method (paper Section 3.2 and Appendix E):
    * Each topological feature with persistence p = d - b is treated as a
      fully observed survival time.  Persistence values are pooled within each
      group, and the persistence survival function S(t) = P(p > t) is
      compared between groups with the log-rank test.
    * The log-rank statistic is computed per homology dimension (stratum) and
      combined into the stratified statistic
      ``chi^2 = (sum_dims (O_d - E_d))^2 / sum_dims V_d``, where ``O_d - E_d``
      is the observed-minus-expected event count and ``V_d`` the
      hypergeometric variance in dimension d (paper Eq. 6-7).
    * The null distribution is obtained by permuting the group labels at the
      *diagram* level (paper Appendix E), which preserves within-diagram
      dependence exactly and is exchangeable under the null; the paper's
      asymptotic chi-square p-value is also available via ``asymptotic=True``.

Degenerate cases (no features in either group, zero variance) are mapped to
p = 1 or to an infinite statistic (reject), respectively.
"""
from __future__ import annotations

import numpy as np
from scipy import stats

from ._common import _perm_groups, _perm_pvalue, _persistence, _validate


def _logrank(e0, e1):
    """Log-rank (O - E, V) for two event-time samples (persistence values)."""
    if len(e0) == 0 and len(e1) == 0:
        return 0.0, 0.0
    e = np.concatenate([e0, e1])
    g = np.concatenate([np.zeros(len(e0)), np.ones(len(e1))])
    if len(e) == 0:
        return 0.0, 0.0
    order = np.argsort(e, kind="mergesort")
    e, g = e[order], g[order]
    _, inv = np.unique(e, return_inverse=True)
    ntimes = int(inv.max()) + 1
    tot = np.bincount(inv, minlength=ntimes)
    d0 = np.bincount(inv[g == 0], minlength=ntimes)
    d1 = tot - d0
    # at risk just before t_j: features with persistence >= t_j
    c0 = np.concatenate([[0], np.cumsum(d0)[:-1]])
    c1 = np.concatenate([[0], np.cumsum(d1)[:-1]])
    n0 = len(e0) - c0
    n1 = len(e1) - c1
    nj = n0 + n1
    dj = tot
    with np.errstate(divide="ignore", invalid="ignore"):
        E = n0 * dj / nj
        V = n0 * n1 * dj * (nj - dj) / (nj * nj * (nj - 1))
    V = np.nan_to_num(V, nan=0.0, posinf=0.0, neginf=0.0)
    Z = float(np.sum(d0 - E))
    Vsum = float(np.sum(V))
    return Z, Vsum


def _stratified_stat(e0_list, e1_list):
    """Stratified log-rank statistic ``Z^2 / V`` over homology dims.

    Returns ``(Z, V)`` so callers can detect the degenerate case V = 0.
    """
    Z = V = 0.0
    for e0, e1 in zip(e0_list, e1_list):
        z, v = _logrank(e0, e1)
        Z += z
        V += v
    return Z, V


def _stat_value(Z, V):
    if V > 0:
        return float(Z * Z / V)
    return 0.0 if Z == 0 else np.inf


def test_strand(diags0, diags1, n_perm=200, seed=None, asymptotic=False):
    """STRAND log-rank two-sample test on pooled feature lifetimes.

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays.
        n_perm: number of diagram-level label permutations (default 200).
        seed: RNG seed for the permutations.
        asymptotic: if True, use the asymptotic chi-square(1) p-value of the
            stratified log-rank statistic instead of the permutation null.

    Returns:
        float p-value in [0, 1].
    """
    d0, d1, nd = _validate(diags0, diags1)
    n0, n1 = len(d0), len(d1)
    N = n0 + n1
    pooled = d0 + d1
    rng = np.random.default_rng(seed)

    # per-sample, per-dim feature lifetimes
    events = [[_persistence(d[dim]) for d in pooled] for dim in range(nd)]
    total = sum(len(e) for lst in events for e in lst)
    if total == 0:
        return 1.0

    Z, V = _stratified_stat(
        [np.concatenate(ev[:n0]) for ev in events],
        [np.concatenate(ev[n0:]) for ev in events])
    obs = _stat_value(Z, V)

    if asymptotic:
        p = 1.0 - stats.chi2.cdf(obs, df=1)
        if V == 0:
            return 0.0 if Z != 0 else 1.0
        return float(np.clip(p, 0.0, 1.0))

    perms = _perm_groups(N, n0, n_perm, rng)
    null = np.empty(n_perm)
    for k in range(n_perm):
        g = perms[k]
        Z, V = _stratified_stat(
            [np.concatenate([ev[i] for i in np.flatnonzero(g)]) for ev in events],
            [np.concatenate([ev[i] for i in np.flatnonzero(~g)]) for ev in events])
        null[k] = _stat_value(Z, V)
    return _perm_pvalue(obs, null, "greater")


In [ ]:
%%writefile tda2s/benchmarks/moon_lazar.py
"""Moon & Lazar (2023) two-stage test on persistence images, as published.

Reference: C. Moon and N. A. Lazar, "Hypothesis testing for shapes using
vectorized persistence diagrams", J. R. Stat. Soc. Ser. C 72(3):628-648 (2023),
doi:10.1093/jrsssc/qlad024. Preprint arXiv:2006.05466, from which the section
and algorithm numbers below are taken. Paper accessed in full; no author code
was released, so this is a direct transcription of Algorithm 1.

Vectorisation (paper Section 2.2)
---------------------------------
A diagram ``PD = {(birth, death)}`` is transformed to ``PD_t = {(u = birth,
v = death - birth)}``. With a Gaussian smoothing function

    f_{(u,v)}(x, y | h) = 1/(2 pi h^2) exp(-((x-u)^2 + (y-v)^2) / (2 h^2))

and a weight ``w(u, v)``, the persistence surface is
``rho(x, y) = sum_{(u,v) in PD_t} f_{(u,v)}(x, y) w(u, v)`` and the persistence
image is the integral of ``rho`` over each pixel. Because ``f`` is a product of
two one-dimensional Gaussians, that integral is computed here in closed form as
a product of normal-CDF differences rather than by quadrature.

Weights offered by the paper: ``"constant"`` (``w = 1``), ``"arctan"``
(``w = arctan(R v^S)``, with the paper's ``R = S = 0.5``) and ``"linear"``
(``w = v``). Section 4.1's method comparison uses 40x40 pixels, ``h = 0.5`` and
the constant weight.

Two-stage procedure (paper Algorithm 1, Sections 3.1-3.2)
---------------------------------------------------------
1. Pre-filter: keep only pixels with ``v_x >= v_y`` (Algorithm 1 line 2). The
   complementary triangle corresponds to an empty region of the transformed
   diagram, leaving ``m(m+1)/2`` of the ``m^2`` pixels.
2. Stage I -- for each surviving pixel compute the *overall* (pooled-across-
   groups) sample standard deviation as the filter statistic (Section 3.1)::

       s^i = sqrt( sum_j sum_k (x^i_{(j,k)} - xbar^i)^2 / (n_1 + n_2 - 1) )

   and drop pixels whose filter statistic is at or below the ``C``-th
   percentile. The filter statistic is deliberately independent of the stage-II
   statistic (Bourgon et al. 2010), which is what keeps the conditional null
   valid.
3. Stage II -- a pooled-variance two-sample t-test per surviving pixel, then a
   multiple-testing adjustment (BH by default; BY also provided).

The returned scalar is the smallest adjusted p-value, so "reject at level
``alpha``" is exactly ``p <= alpha`` under the chosen FDR rule.
"""
from __future__ import annotations

import numpy as np
from scipy import stats

from ._common import _points, _validate

__all__ = ["test_moon_lazar", "persistence_images"]

_WEIGHTS = {
    "constant": lambda v: np.ones_like(v),
    "arctan": lambda v: np.arctan(0.5 * np.power(np.abs(v), 0.5)),
    "linear": lambda v: v,
}


def _bh_adjusted(pvals):
    """Benjamini-Hochberg adjusted p-values (monotone), shape = input."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p, kind="mergesort")
    q = p[order] * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    adj = np.empty(m)
    adj[order] = np.clip(q, 0.0, 1.0)
    return adj


def _by_adjusted(pvals):
    """Benjamini-Yekutieli adjusted p-values (BH scaled by the harmonic sum)."""
    m = len(pvals)
    c_m = np.sum(1.0 / np.arange(1, m + 1))
    return np.clip(_bh_adjusted(pvals) * c_m, 0.0, 1.0)


_ADJUST = {"bh": _bh_adjusted, "by": _by_adjusted}


def persistence_images(diagrams, resolution=40, bandwidth=0.5, weight="constant",
                       im_range=None):
    """Persistence images of a list of diagrams (Adams et al. parameterisation).

    Args:
        diagrams: list over samples of ``(k, 2)`` (birth, death) arrays -- ONE
            homology dimension.
        resolution: ``m``; the image is ``m x m`` pixels.
        bandwidth: Gaussian smoothing ``h``.
        weight: ``"constant"``, ``"arctan"`` (R = S = 0.5) or ``"linear"``.
        im_range: ``(lo, hi)`` shared by the birth and persistence axes;
            defaults to ``(0, max persistence-or-birth over the pooled input)``.
            A common square range is required because the paper's pre-filter
            compares the two pixel coordinates directly.

    Returns:
        ``(images, vx, vy)``: ``images`` of shape ``(n_samples, m*m)`` in
        row-major (x, y) order, plus the pixel-centre coordinate vectors.
    """
    try:
        wfn = _WEIGHTS[weight]
    except KeyError:
        raise ValueError(
            f"weight must be one of {sorted(_WEIGHTS)}, got {weight!r}") from None

    pts = [_points(d) for d in diagrams]
    trans = [np.column_stack([p[:, 0], p[:, 1] - p[:, 0]]) if len(p)
             else np.empty((0, 2)) for p in pts]

    if im_range is None:
        hi = 0.0
        for t in trans:
            if len(t):
                hi = max(hi, float(t.max()))
        im_range = (0.0, hi if hi > 0 else 1.0)
    lo, hi = float(im_range[0]), float(im_range[1])

    edges = np.linspace(lo, hi, resolution + 1)
    centres = 0.5 * (edges[:-1] + edges[1:])
    vx = np.repeat(centres, resolution)      # x varies slowest (row-major)
    vy = np.tile(centres, resolution)

    out = np.zeros((len(trans), resolution * resolution))
    for i, t in enumerate(trans):
        if not len(t):
            continue
        u, v = t[:, 0], t[:, 1]
        w = wfn(v)
        # exact cell integral: product of 1-D normal-CDF differences
        cx = stats.norm.cdf(edges[None, :], loc=u[:, None], scale=bandwidth)
        cy = stats.norm.cdf(edges[None, :], loc=v[:, None], scale=bandwidth)
        px = np.diff(cx, axis=1)             # (n_pts, resolution)
        py = np.diff(cy, axis=1)
        out[i] = ((w[:, None] * px)[:, :, None] * py[:, None, :]).sum(axis=0).ravel()
    return out, vx, vy


def test_moon_lazar(diags0, diags1, dim_index=-1, resolution=40, bandwidth=0.5,
                    weight="constant", filter_threshold=80.0, adjust="bh",
                    im_range=None, alpha=0.05):
    """Two-stage (filter + pooled-variance t-test + FDR) test on persistence images.

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays.
        dim_index: index into each sample's per-dimension diagram list
            (NOT the homology degree). ``-1`` is the last entry, which is the
            highest homology dimension present -- H1 for the usual ``(0, 1)``
            lists and for bare H1 lists alike. The paper's simulations use
            dimension one.
        resolution: persistence-image side length ``m`` (paper: 40).
        bandwidth: Gaussian smoothing ``h`` (paper Section 4.1: 0.5).
        weight: ``"constant"`` (paper Section 4.1), ``"arctan"`` or ``"linear"``.
        filter_threshold: ``C`` in percent; pixels at or below the ``C``-th
            percentile of the stage-I filter statistic are dropped (paper: 80).
        adjust: ``"bh"`` (paper's default) or ``"by"``.
        im_range: shared ``(lo, hi)`` for both image axes.
        alpha: retained for interface symmetry; the decision rule is
            ``returned p <= alpha``.

    Returns:
        float: smallest adjusted p-value over surviving pixels, in [0, 1].
        Returns 1.0 when no pixel survives the filter.
    """
    d0, d1, nd = _validate(diags0, diags1)
    if not -nd <= dim_index < nd:
        raise ValueError(
            f"dim_index {dim_index} out of range for {nd}-dim diagram lists")
    try:
        adjust_fn = _ADJUST[adjust]
    except KeyError:
        raise ValueError(
            f"adjust must be one of {sorted(_ADJUST)}, got {adjust!r}") from None

    pooled = [d[dim_index] for d in d0] + [d[dim_index] for d in d1]
    V, vx, vy = persistence_images(pooled, resolution=resolution,
                                   bandwidth=bandwidth, weight=weight,
                                   im_range=im_range)
    n0, n1 = len(d0), len(d1)

    keep = vx >= vy                                   # Algorithm 1, line 2
    V = V[:, keep]
    if V.shape[1] == 0:
        return 1.0

    # Stage I: overall (pooled) sample sd per pixel, Section 3.1
    xbar = V.mean(axis=0)
    s = np.sqrt(((V - xbar) ** 2).sum(axis=0) / (n0 + n1 - 1))
    t_C = np.percentile(s, filter_threshold)
    keep2 = s > t_C
    if not keep2.any():
        return 1.0
    V = V[:, keep2]

    # Stage II: pooled-variance two-sample t-test per surviving pixel
    with np.errstate(invalid="ignore", divide="ignore"):
        _, ps = stats.ttest_ind(V[:n0], V[n0:], axis=0, equal_var=True)
    ps = np.asarray(ps, dtype=float)
    ps[~np.isfinite(ps)] = 1.0                        # constant pixels

    return float(np.clip(adjust_fn(ps).min(), 0.0, 1.0))


In [ ]:
%%writefile tda2s/benchmarks/frechet_anova.py
"""Frechet ANOVA test of Dubey & Muller (2019), as published.

Reference: P. Dubey and H.-G. Muller, "Frechet analysis of variance for random
objects", Biometrika 106(4):803-821 (2019), doi:10.1093/biomet/asz052.
Preprint arXiv:1710.02761v3, from which the equation numbers below are taken.
Paper accessed in full; no author code was released, so this is a direct
transcription of Section 4.

Method (paper Section 4, equations 6-13)
----------------------------------------
For groups ``G_1, ..., G_k`` of sizes ``n_j`` in a bounded metric space
``(Omega, d)``, with ``lambda_j = n_j / n``:

* group Frechet mean and variance (page 6)::

      mu_j   = argmin_w (1/n_j) sum_{i in G_j} d^2(w, Y_i)
      V_j    = (1/n_j) sum_{i in G_j} d^2(mu_j, Y_i)

* the variance estimate of eq. (3), applied per group::

      s2_j   = (1/n_j) sum_{i in G_j} d^4(mu_j, Y_i)
               - [ (1/n_j) sum_{i in G_j} d^2(mu_j, Y_i) ]^2

* pooled Frechet mean and variance, eq. (6)::

      mu_p   = argmin_w (1/n) sum_j sum_{i in G_j} d^2(w, Y_i)
      V_p    = (1/n) sum_j sum_{i in G_j} d^2(mu_p, Y_i)

* the two auxiliary statistics, eqs. (7) and (8)::

      F_n    = V_p - sum_j lambda_j V_j
      U_n    = sum_{j < l} (lambda_j lambda_l) / (s2_j s2_l) * (V_j - V_l)^2

* the test statistic, eq. (11)::

      T_n    = n U_n / sum_j (lambda_j / s2_j)
               + n F_n^2 / sum_j (lambda_j^2 s2_j)

Under the null of equal Frechet means *and* variances, ``T_n -> chi^2_(k-1)``
(Theorem 2, eq. 12) and the level-alpha rejection region is
``T_n > chi^2_{k-1, alpha}`` (eq. 13). ``F_n`` targets mean differences and
``U_n`` variance differences, so the test has power against both -- the paper
notes ``U_n`` is a Levene-type term, which is why a pure between/within ratio
would be blind to the scale alternatives of its Figure 1 (right panel).

The paper adds (end of Section 4) that "asymptotic tests may not work very well
... where the group sample sizes are small", and that permutation tests using
``T_n`` give more accurate level-alpha tests; ``n_perm`` selects that route and
is the default here, because two-sample topology problems are small-n.

Metric space
------------
Dubey-Muller is stated for any bounded metric space, so the caller chooses one:

* ``space="summary"`` (default) embeds each diagram in ``L^2`` via a functional
  summary (Betti curves by default, concatenated over homology dimensions).
  There the Frechet mean is the arithmetic mean -- exact, unique, closed-form.
  This is also the space P1's own scope-limit section argues for, since
  ``(D_p, W_p)`` has non-unique Frechet means (Turner et al.; Che et al.).
* ``space="diagram"`` uses ``W_2`` between diagrams and needs a true Wasserstein
  barycentre. That requires the ``POT`` package via ``gudhi.wasserstein``; if it
  is missing the call raises rather than silently substituting a proxy.
"""
from __future__ import annotations

import numpy as np
from scipy import stats

from ._common import _perm_groups, _perm_pvalue, _points, _validate

__all__ = ["test_frechet_anova", "frechet_anova_statistic"]


def frechet_anova_statistic(sq_dists_to_group_mean, sq_dists_to_pooled_mean, labels):
    """Dubey-Muller ``T_n`` from precomputed squared distances (eqs. 6-11).

    Args:
        sq_dists_to_group_mean: ``(n,)`` array; entry ``i`` is ``d^2(mu_{g(i)},
            Y_i)``, the squared distance from unit ``i`` to *its own* group's
            Frechet mean.
        sq_dists_to_pooled_mean: ``(n,)`` array of ``d^2(mu_p, Y_i)``.
        labels: ``(n,)`` integer group labels.

    Returns:
        float ``T_n``. ``inf`` if a group's variance estimate ``s2_j`` is zero
        (degenerate group: reject).
    """
    a = np.asarray(sq_dists_to_group_mean, dtype=float)
    b = np.asarray(sq_dists_to_pooled_mean, dtype=float)
    labels = np.asarray(labels)
    groups = np.unique(labels)
    n = labels.size

    lam, V, s2 = [], [], []
    for g in groups:
        m = labels == g
        d2 = a[m]
        lam.append(m.sum() / n)
        V.append(d2.mean())
        s2.append((d2 ** 2).mean() - d2.mean() ** 2)     # eq. (3) per group
    lam = np.asarray(lam)
    V = np.asarray(V)
    s2 = np.asarray(s2)

    if np.any(s2 <= 0) or not np.all(np.isfinite(s2)):
        return np.inf

    V_p = b.mean()                                        # eq. (6)
    F_n = V_p - float(lam @ V)                            # eq. (7)

    U_n = 0.0                                             # eq. (8)
    for j in range(len(groups)):
        for l in range(j + 1, len(groups)):
            U_n += (lam[j] * lam[l]) / (s2[j] * s2[l]) * (V[j] - V[l]) ** 2

    term_u = n * U_n / float(np.sum(lam / s2))            # eq. (11), first term
    term_f = n * F_n ** 2 / float(np.sum(lam ** 2 * s2))  # eq. (11), second term
    return float(term_u + term_f)


def _l2_summary_stat(X, labels):
    """``T_n`` in ``L^2``: the Frechet mean is the arithmetic mean."""
    X = np.asarray(X, dtype=float)
    a = np.empty(len(X))
    for g in np.unique(labels):
        m = labels == g
        a[m] = ((X[m] - X[m].mean(axis=0)) ** 2).sum(axis=1)
    b = ((X - X.mean(axis=0)) ** 2).sum(axis=1)
    return frechet_anova_statistic(a, b, labels)


def _summary_vectors(diags0, diags1, representation, n_points, interval):
    """Embed every diagram list in ``R^p`` via a functional summary."""
    from tda2s.vec import vectorise

    d0, d1, _ = _validate(diags0, diags1)
    pooled = d0 + d1
    if interval is None:
        hi = 0.0
        for d in pooled:
            for dgm in d:
                p = _points(dgm)
                if len(p):
                    hi = max(hi, float(p[:, 1].max()))
        interval = (0.0, hi if hi > 0 else 1.0)
    rows = []
    for d in pooled:
        clean = [_points(dgm) for dgm in d]
        v = vectorise(clean, representation, interval=interval, n_points=n_points)
        rows.append(np.ravel(np.asarray(v, dtype=float)))
    return np.stack(rows), len(d0), len(d1)


def _diagram_barycentre_stat(diags0, diags1, labels, order):
    """``T_n`` in ``(D_2, W_2)`` using a true Wasserstein barycentre."""
    try:
        from gudhi.wasserstein import wasserstein_distance
        from gudhi.wasserstein.barycenter import lagrangian_barycenter
    except ImportError as exc:                            # pragma: no cover
        raise ImportError(
            "space='diagram' needs a Wasserstein barycentre, which requires the "
            "optional POT package (`pip install pot`). Dubey-Muller's Frechet "
            "mean has no closed form in (D_p, W_p); this wrapper will not "
            "substitute a proxy. Use space='summary' instead."
        ) from exc

    d0, d1, nd = _validate(diags0, diags1)
    pooled = d0 + d1

    def _mean_and_sq(idx):
        """Barycentre over the given units, and each unit's squared W_2 to it."""
        sq = np.zeros(len(pooled))
        for dim in range(nd):
            dgms = [_points(pooled[i][dim]) for i in idx]
            bary = lagrangian_barycenter(pdiagset=dgms, init=0)
            if bary is None:
                bary = np.empty((0, 2))
            for i in range(len(pooled)):
                w = wasserstein_distance(_points(pooled[i][dim]), bary,
                                         order=order, internal_p=2)
                sq[i] += float(w) ** 2
        return sq

    a = np.empty(len(pooled))
    for g in np.unique(labels):
        idx = np.flatnonzero(labels == g)
        a[idx] = _mean_and_sq(idx)[idx]
    b = _mean_and_sq(np.arange(len(pooled)))
    return frechet_anova_statistic(a, b, labels)


def test_frechet_anova(diags0, diags1, space="summary", representation="betti",
                       n_points=100, interval=None, n_perm=1000, seed=None,
                       order=2):
    """Dubey-Muller Frechet ANOVA two-sample test on persistence diagrams.

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays.
        space: ``"summary"`` (L^2 functional summary, exact Frechet mean) or
            ``"diagram"`` (W_2 on diagrams, needs a barycentre via POT).
        representation: summary used when ``space="summary"``; any name accepted
            by ``tda2s.vec.vectorise`` (default ``"betti"``).
        n_points, interval: grid for the summary.
        n_perm: permutations for the label-permutation null. ``None`` uses the
            asymptotic ``chi^2_{k-1}`` of Theorem 2 instead -- the paper warns
            that route is unreliable at small group sizes.
        seed: RNG seed for the permutations.
        order: Wasserstein order when ``space="diagram"``.

    Returns:
        float p-value in [0, 1].
    """
    if space == "summary":
        X, n0, n1 = _summary_vectors(diags0, diags1, representation, n_points,
                                     interval)
        stat = lambda lab: _l2_summary_stat(X, lab)
    elif space == "diagram":
        d0, d1, _ = _validate(diags0, diags1)
        n0, n1 = len(d0), len(d1)
        stat = lambda lab: _diagram_barycentre_stat(diags0, diags1, lab, order)
    else:
        raise ValueError(f"space must be 'summary' or 'diagram', got {space!r}")

    N = n0 + n1
    labels = np.zeros(N, dtype=int)
    labels[:n0] = 1
    obs = stat(labels)

    if n_perm is None:
        if not np.isfinite(obs):
            return 0.0
        return float(stats.chi2.sf(obs, df=1))            # eq. (12), k = 2

    rng = np.random.default_rng(seed)
    perms = _perm_groups(N, n0, n_perm, rng)
    null = np.array([stat(perms[b].astype(int)) for b in range(n_perm)])
    return _perm_pvalue(obs, null, "greater")


In [ ]:
%%writefile tda2s/benchmarks/krebs_rademacher.py
"""Krebs & Rademacher (2024) relevant-difference test on persistence diagrams.

Reference: J. Krebs and D. Rademacher, "Two-sample tests for relevant
differences in persistence diagrams", arXiv:2401.10349v1 (2024). Equation
numbers below are the paper's. Paper accessed in full; no author code was
released and the paper contains no simulation study, so this is a direct
transcription of Section 1.2 with no published figure or table to reproduce.

What the paper actually tests
-----------------------------
NOT a difference of mean summaries: the paper compares *dispersion* of the two
diagram populations under a Wasserstein metric, in a **relevant-difference**
framing with an externally set tolerance ``Delta >= 0`` (eq. 1.7)::

    H_0: (sigma^2_X - sigma^2_Y)^2 <= Delta   vs   H_1: (...)^2 > Delta

Section 1.1 does this with Frechet variances, which needs a Frechet mean in
``(D_r, W_r)`` -- a non-unique, expensive optimisation. Section 1.2 uses the
*independent copy* ("inco") variance instead, which needs no mean at all
(eq. 1.4)::

    sigma^2(X) := (1/2) E[ W_r^2( PD(X_0), PD(X') ) ]

estimated by the U-statistic with kernel ``h(.,.) = 2^{-1} W_r^2(PD(.), PD(.))``
(eqs. 1.5, 1.13). This module implements the Section 1.2 (inco-variance) test,
which the paper itself recommends as the practical route.

Statistic (eqs. 1.11-1.16)
--------------------------
Two-parameter partial-sum processes, normalised by the FULL sample size so that
``sigma^2_X(s, t) ~ s t sigma^2_X`` (eq. 1.12)::

    sigma^2_X(s, t) = 1/(m(m-1)) sum_{i<=floor(ms)} sum_{j<=floor(mt), j!=i} h(X_i, X_j)

with ``D(s, t) = sigma^2_X(s, t) - sigma^2_Y(s, t)`` (eq. 1.11) and
``D_hat = D(1, 1)``. The self-normaliser (eq. 1.14) is::

    V_hat = { int int [ D(s,t)^2 - (s t D_hat)^2 ]^2 nu(ds, dt) }^{1/2}

and the test statistic and rejection rule are::

    W_hat = (D_hat^2 - Delta) / V_hat ,      reject if W_hat > q_{1-alpha}

where ``q_{1-alpha}`` is the quantile of the pivotal limit (eq. 1.16)::

    W = 2 B(1) / { int int [ s t ( t B(s) + s B(t) - 2 s t B(1) ) ]^2 dnu(s,t) }^{1/2}

for a standard Brownian motion ``B``. The ``sqrt(m+n)`` rates of Theorem 1.4
cancel in the ratio, so no explicit scaling appears. ``nu`` is any probability
measure on ``[0, 1]^2``; the uniform measure on a regular grid is used here and
is exposed via ``grid``.

The Delta = 0 boundary
----------------------
The paper's framework is built for ``Delta > 0``. At ``Delta = 0`` its limit
degenerates: Theorem 1.4's scale factor is
``xi = 2 sqrt(Gamma_X/tau + Gamma_Y/(1-tau)) (sigma^2_X - sigma^2_Y)``, which is
zero exactly when the null holds with ``Delta = 0``, so ``W`` is not the right
reference law there. Because a benchmark suite needs every competitor evaluated
at a common classical null, ``delta=0`` (the default) instead calibrates
``D_hat^2`` by label permutation. That path is an extension, not the paper's
test, and is labelled as such; pass ``delta > 0`` for the published procedure.
"""
from __future__ import annotations

import numpy as np

from ._common import _perm_groups, _perm_pvalue, _points, _validate

__all__ = ["test_krebs_rademacher", "inco_variance", "pivotal_quantiles"]


def _pairwise_kernel(diagrams, metric):
    """``K[i, j] = (1/2) W_r^2(PD_i, PD_j)``, the kernel of eq. (1.13)."""
    n = len(diagrams)
    K = np.zeros((n, n))
    if metric == "wasserstein":
        import persim
        dist = lambda a, b: float(persim.wasserstein(a, b))
    elif metric == "bottleneck":
        import gudhi as gd
        dist = lambda a, b: float(gd.bottleneck_distance(a, b))
    else:
        raise ValueError(f"metric must be 'wasserstein' or 'bottleneck', got {metric!r}")
    pts = [_points(d) for d in diagrams]
    for i in range(n):
        for j in range(i + 1, n):
            K[i, j] = K[j, i] = 0.5 * dist(pts[i], pts[j]) ** 2
    return K


def inco_variance(K):
    """Inco-variance U-statistic ``sigma^2 = K.sum() / (n(n-1))`` (eq. 1.5)."""
    n = K.shape[0]
    if n < 2:
        return 0.0
    return float(K.sum() / (n * (n - 1)))


def _partial_sums(K, grid):
    """``sigma^2(s, t)`` of eq. (1.12) on a regular ``grid x grid`` mesh."""
    n = K.shape[0]
    if n < 2:
        return np.zeros((grid, grid))
    # cumulative block sums; K has a zero diagonal, so no diagonal correction
    C = np.zeros((n + 1, n + 1))
    C[1:, 1:] = K.cumsum(axis=0).cumsum(axis=1)
    ks = np.floor(n * np.arange(1, grid + 1) / grid).astype(int)
    return C[np.ix_(ks, ks)] / (n * (n - 1))


def _self_normaliser(D_st, D_hat, grid):
    """``V_hat`` of eq. (1.14) under uniform ``nu`` on the grid."""
    s = np.arange(1, grid + 1) / grid
    st = np.outer(s, s)
    integrand = (D_st ** 2 - (st * D_hat) ** 2) ** 2
    return float(np.sqrt(integrand.mean()))


def _simulate_W(grid, n_draws, rng):
    """Draws from the pivotal limit ``W`` of eq. (1.16)."""
    s = np.arange(1, grid + 1) / grid
    st = np.outer(s, s)
    incr = rng.normal(0.0, np.sqrt(1.0 / grid), size=(n_draws, grid))
    B = np.cumsum(incr, axis=1)                       # B(s_k), B(1) = B[:, -1]
    B1 = B[:, -1]
    # t B(s) + s B(t) - 2 s t B(1), then scaled by s t
    term = (s[None, None, :] * B[:, :, None]          # t B(s): rows = s, cols = t
            + s[None, :, None] * B[:, None, :]        # s B(t)
            - 2.0 * st[None] * B1[:, None, None])
    denom = np.sqrt(((st[None] * term) ** 2).mean(axis=(1, 2)))
    return 2.0 * B1 / denom


def pivotal_quantiles(q, grid=20, n_draws=20000, seed=0):
    """Quantiles of the pivotal limit ``W`` of eq. (1.16).

    Args:
        q: quantile level(s) in (0, 1).
        grid: mesh size of the uniform ``nu`` on ``[0, 1]^2`` (must match the
            grid used for ``V_hat``).
        n_draws: Monte Carlo Brownian paths.
        seed: RNG seed.

    Returns:
        Array of quantiles of ``W``.
    """
    return np.quantile(_simulate_W(grid, n_draws, np.random.default_rng(seed)), q)


def test_krebs_rademacher(diags0, diags1, delta=0.0, dim_index=-1,
                          metric="wasserstein", grid=20, n_perm=200, seed=None,
                          n_pivotal=20000):
    """Krebs-Rademacher inco-variance test (paper Section 1.2).

    Args:
        diags0, diags1: lists (over samples) of lists (per homology dim) of
            ``(k, 2)`` birth-death arrays. Sample ORDER matters: the partial-sum
            processes of eq. (1.12) are taken in the given order (irrelevant for
            i.i.d. samples, meaningful for the paper's time-series setting).
        delta: relevance tolerance ``Delta`` of eq. (1.7). ``> 0`` runs the
            published self-normalised test; ``0`` (default) falls back to a
            permutation null -- see the module docstring.
        dim_index: index into each sample's per-dimension diagram list
            (NOT the homology degree); ``-1`` is the last, i.e. the highest
            homology dimension present. The paper fixes one dimension.
        metric: ``"wasserstein"`` (``W_1`` via persim) or ``"bottleneck"``
            (``W_inf`` via gudhi).
        grid: mesh size for ``nu`` on ``[0, 1]^2``.
        n_perm: permutations used when ``delta == 0``.
        seed: RNG seed.
        n_pivotal: Brownian draws for the eq. (1.16) reference law.

    Returns:
        float p-value in [0, 1].
    """
    d0, d1, nd = _validate(diags0, diags1)
    if not -nd <= dim_index < nd:
        raise ValueError(
            f"dim_index {dim_index} out of range for {nd}-dim diagram lists")
    m, n = len(d0), len(d1)
    if m < 2 or n < 2:
        return 1.0

    pooled = [d[dim_index] for d in d0] + [d[dim_index] for d in d1]
    K = _pairwise_kernel(pooled, metric)
    idx0 = np.arange(m)
    idx1 = np.arange(m, m + n)

    def _d_hat(i0, i1):
        return inco_variance(K[np.ix_(i0, i0)]) - inco_variance(K[np.ix_(i1, i1)])

    if delta > 0:
        sx = _partial_sums(K[np.ix_(idx0, idx0)], grid)
        sy = _partial_sums(K[np.ix_(idx1, idx1)], grid)
        D_st = sx - sy
        D_hat = float(D_st[-1, -1])
        V_hat = _self_normaliser(D_st, D_hat, grid)
        if V_hat <= 0:
            return 1.0
        W_hat = (D_hat ** 2 - delta) / V_hat                  # eq. (1.8) form
        W = _simulate_W(grid, n_pivotal, np.random.default_rng(0 if seed is None else seed))
        return float((1.0 + np.sum(W >= W_hat)) / (1.0 + W.size))

    # delta == 0: permutation calibration of D_hat^2 (extension, see docstring)
    obs = _d_hat(idx0, idx1) ** 2
    rng = np.random.default_rng(seed)
    perms = _perm_groups(m + n, m, n_perm, rng)
    null = np.empty(n_perm)
    for b in range(n_perm):
        g = perms[b]
        null[b] = _d_hat(np.flatnonzero(g), np.flatnonzero(~g)) ** 2
    return _perm_pvalue(obs, null, "greater")


In [ ]:
%%writefile tda2s/adapters/__init__.py
"""Adapters over ``tcda_uq``: P1's import surface for the shared estimators.

See ``docs/reuse_from_tcda_uq.md`` for the full reuse audit and boundary
(CP_TATE owns bands; P1 owns p-values; the shared objects are the AIPW curve
and the per-unit EIF process).
"""

from .tcda_uq import aipw_curve, ctate_learner, silhouettes, tri_oracle

__all__ = ["aipw_curve", "silhouettes", "tri_oracle", "ctate_learner"]


In [ ]:
%%writefile tda2s/adapters/tcda_uq.py
"""Thin shim over ``tcda_uq``: import surface for P1, zero reimplemented math.

P1 reuses the released CP_TATE library (``tcda_uq``, installed from git) for
AIPW estimation, cross-fitting, the functional DR-learner, silhouettes, and the
tri-oracle simulation. This module is pure delegation and argument plumbing;
every function forwards to ``tcda_uq`` and only reshapes return values where the
P1 test statistic needs a different layout (e.g. stacking per-dimension score
lists into one array).

Boundary (see ``docs/reuse_from_tcda_uq.md``): tcda_uq owns confidence /
prediction bands; P1 owns p-values. The only shared objects are the AIPW curve
(``aipw[d]``) and the per-unit efficient-influence-function process
(``scores``), from which P1 computes its test statistic and multiplier-bootstrap
null law. Bands are deliberately not exposed here.
"""

from __future__ import annotations

from typing import Tuple

import numpy as np


def aipw_curve(sample, tseq, n_basis: int, n_folds: int = 5, **cross_fit_kwargs) -> dict:
    """Cross-fitted AIPW estimate of the TATE curve(s) for one sample.

    Delegates to ``tcda_uq.estimators.cross_fit`` and reshapes the result for
    P1's test statistic:

    * ``aipw``: list, one ``[resolution]`` mean AIPW curve per homology dim.
    * ``scores``: ``(n, n_hom_dim, resolution)`` per-unit doubly-robust score
      process (the cross-fitted EIF; its mean over units is ``aipw``).
    * ``pi_hat``: ``(n,)`` cross-fitted propensity.
    * ``tseq``: the silhouette grid.

    Args:
        sample: observed triplet ``(phi, A, X)`` with ``phi`` ``[n, n_hom_dim,
            resolution]``, ``A`` ``[n]``, ``X`` ``[n, d]``.
        tseq: silhouette grid ``[resolution]``.
        n_basis: Fourier basis size for the outcome regression.
        n_folds: number of cross-fitting folds (``cross_fit``'s ``n_splits``).
        **cross_fit_kwargs: forwarded verbatim to ``cross_fit`` (e.g.
            ``propensity_estimator``, ``propensity_feature_fn``, ``stratify``,
            ``random_state``); default ``None`` reproduces tcda_uq defaults.
    """
    from tcda_uq.estimators import cross_fit

    result = cross_fit(sample, tseq, n_basis=n_basis, n_splits=n_folds,
                       **cross_fit_kwargs)
    return {
        "aipw": result.aipw,
        "scores": np.stack(result.scores, axis=1),
        "pi_hat": result.pi_hat,
        "tseq": np.asarray(result.tseq),
    }


def silhouettes(diagrams, interval=(0.0, 0.2), r: float = 3.0,
                resolution: int = 100) -> np.ndarray:
    """Power-weighted silhouettes of persistence diagrams.

    Delegates to ``tcda_uq.silhouette.compute_silhouette`` (defaults: interval
    ``(0, 0.2)``, ``r=3``, ``resolution=100``). Returns ``(n_hom_dim,
    resolution)``.
    """
    from tcda_uq.silhouette import compute_silhouette

    return compute_silhouette(diagrams, interval=interval, r=r,
                              resolution=resolution)


def tri_oracle(n: int, **kwargs) -> "SimulationSample":
    """Draw a sample from the tri-oracle simulation.

    Delegates to ``tcda_uq.datasets.TriOracleSimulation``: ``kwargs`` are
    passed to its constructor (``n_cov``, ``n_hom_dim``, ``resolution``,
    ``interval``, ``n_basis``, ``noise_scale``, ``seed``, ...), then ``sample(n)``
    is drawn. Returns a ``SimulationSample`` with ``oracle_tate``,
    ``oracle_ctate``, ``oracle_itte`` and the ``.observed`` triplet.
    """
    from tcda_uq.datasets import TriOracleSimulation

    return TriOracleSimulation(**kwargs).sample(n)


def ctate_learner(*args, **kwargs):
    """Functional DR-learner for the CTATE (same signature as ``CTATEDRLearner``).

    Pure delegation to ``tcda_uq.estimators.CTATEDRLearner``; the returned
    object is a ``CTATEDRLearner``: ``fit(sample, tseq, cross_fit_result=None,
    **cross_fit_kwargs)`` then ``predict(X_eval)``.
    """
    from tcda_uq.estimators import CTATEDRLearner

    return CTATEDRLearner(*args, **kwargs)


In [ ]:
%%writefile tda2s/adapters/dr_test.py
"""Phase 2 prototype DR test of H0^out (task 2.4); polished in Phase 3.

The test statistic and its null follow the boundary statement of
``docs/reuse_from_tcda_uq.md``: P1's statistic is

    T_n = sqrt(n) * max_d || psi_hat_d ||_inf,

computed from ``cross_fit(...).aipw[d]`` (the cross-fitted AIPW TATE curve), and
its null is the multiplier bootstrap over the *centered* per-unit EIF process
``cross_fit(...).scores[d] - mean``:

    G_b(t) = n^{-1/2} sum_i xi_i (s_{i,d}(t) - mean_d(t)),  xi_i ~ N(0, 1),

compared at ``max_d sup_t |G_b(t)|`` (``tda2s.resample.multiplier_bootstrap``
returns exactly these sup draws). The multipliers ``xi_i`` are drawn once per
unit and shared across the homology degrees, since the degrees are dependent
functionals of the same units; see the note in
:func:`prototype_dr_from_phi`. This is a prototype: Phase 3 calibrates the
statistic (multiplicity handling, learner sweep, stratified-permutation null)
and relocates it to the Phase 3 deliverable; nothing here reimplements AIPW,
cross-fitting or the DR-learner, which are imported from ``tcda_uq`` through the
``tda2s.adapters.tcda_uq.aipw_curve`` shim.
"""

from __future__ import annotations

import numpy as np

from tda2s.resample import multiplier_bootstrap


def _default_rf(seed):
    """Seeded random-forest propensity estimator (default in tcda_uq)."""
    from sklearn.ensemble import RandomForestClassifier
    return RandomForestClassifier(random_state=int(seed))


def prototype_dr_from_phi(phi, A, X, tseq, n_basis=8, n_folds=2, n_draws=2000,
                          seed=0, **cross_fit_kwargs) -> float:
    """Prototype DR p-value from a silhouette triplet ``(phi, A, X)``.

    Splits ``prototype_dr_pvalue`` at the silhouette stage so sweeps that
    share the clouds across group assignments (Phase 2.3 common-random-numbers
    design) compute ``phi`` once and call this per assignment.

    Args:
        phi: ``(n, n_hom_dim, resolution)`` silhouette array (the
            ``phi`` returned by ``tda2s.dgp.to_silhouette_sample``).
        A: ``[n]`` group labels.
        X: ``[n, d_x]`` covariates.
        tseq: ``[resolution]`` grid underlying the silhouettes.
        n_basis: Fourier basis size of the outcome regression.
        n_folds: cross-fitting folds (``cross_fit``'s ``n_splits``).
        n_draws: multiplier-bootstrap draws.
        seed: RNG seed (cross_fit ``random_state`` and the bootstrap stream).
        **cross_fit_kwargs: forwarded to ``cross_fit`` (e.g. a
            ``propensity_estimator``); default reproduces tcda_uq defaults.

    Returns:
        float p-value in [0, 1] for ``H0^out: psi_d = 0``.
    """
    from tda2s.adapters.tcda_uq import aipw_curve

    cross_fit_kwargs.setdefault("propensity_estimator", _default_rf(seed))
    res = aipw_curve((phi, A, X), tseq, n_basis=n_basis, n_folds=n_folds,
                     random_state=seed, **cross_fit_kwargs)
    scores = res["scores"]                     # (n, n_hom_dim, resolution)
    aipw = res["aipw"]                         # list, per dim, [resolution]
    n = int(A.shape[0])

    T_obs = float(np.sqrt(n) * np.max([np.abs(np.asarray(psi)).max() for psi in aipw]))

    # One multiplier draw per *unit*, shared across homology degrees. The
    # degrees are two functionals of the same n units, so their EIF processes
    # are dependent; bootstrapping each degree with its own multipliers would
    # make the null's two components independent, and the maximum of two
    # independent copies stochastically dominates the maximum of positively
    # dependent ones with the same marginals. That inflates the null, and the
    # test would come out conservative -- which is exactly what the gate's size
    # criterion measures. Stacking the degrees along the curve axis and taking
    # one sup over the stack is ``max_d sup_t`` under shared multipliers.
    rng = np.random.default_rng(seed)
    centered = scores - scores.mean(axis=0, keepdims=True)   # (n, n_dim, res)
    stacked = centered.reshape(centered.shape[0], -1)        # (n, n_dim * res)
    nulls = multiplier_bootstrap(stacked, n_draws, rng)
    return float((1.0 + np.count_nonzero(nulls >= T_obs)) / (1.0 + n_draws))


def prototype_dr_pvalue(clouds, X, A, filtration="alpha", homology_dims=(0, 1),
                        interval=(0.0, 2.0), r: float = 3.0, resolution: int = 100,
                        n_basis: int = 8, n_folds: int = 2, n_draws: int = 2000,
                        seed: int = 0, **cross_fit_kwargs) -> float:
    """Prototype doubly-robust test of ``H0^out: psi_d = 0`` (full pipeline).

    Args:
        clouds: list of ``(m_i, d)`` point clouds (one per unit).
        X: ``[n, d_x]`` covariate matrix.
        A: ``[n]`` group labels.
        filtration, homology_dims: passed to ``tda2s.ph.compute_diagrams``.
        interval, r, resolution: silhouette domain, power weight and grid size
            (the same grid must cover the persistence scales of the clouds).
        n_basis: Fourier basis size of the outcome regression.
        n_folds: cross-fitting folds (``cross_fit``'s ``n_splits``).
        n_draws: multiplier-bootstrap draws.
        seed: RNG seed (cross_fit ``random_state`` and the bootstrap stream).
        **cross_fit_kwargs: forwarded to ``cross_fit`` (e.g. a
            ``propensity_estimator``); default reproduces tcda_uq defaults.

    Returns:
        float p-value in [0, 1]: the fraction of multiplier draws with
        ``max_d sup_t |G_b^{(d)}(t)| >= T_n`` (Phipson-Smyth convention).
    """
    from tda2s.dgp import to_silhouette_sample

    phi, A, X = to_silhouette_sample(clouds, X, A, filtration=filtration,
                                     homology_dims=homology_dims,
                                     interval=interval, r=r, resolution=resolution)
    tseq = np.linspace(interval[0], interval[1], resolution)
    return prototype_dr_from_phi(phi, A, X, tseq, n_basis=n_basis, n_folds=n_folds,
                                 n_draws=n_draws, seed=seed, **cross_fit_kwargs)

In [ ]:
%%writefile experiments/phase2_imbalance_sweep.py
"""Phase 2 gate: covariate-shift failure of the field's tests (tasks 2.1-2.4).

Two experiments and Figure 1.

Part A -- false-positive sweep (task 2.3), Figure 1a:
  DGP: ``CloudSampleDGP(group_effect=0)``; X ~ N(0, I_3); propensity
  ``expit(PROP_SCALE * lambda * X @ beta)``; topology ``k(x) =
  1 + floor(expit(x_0) * 3)`` loops.  For lambda = 0 the groups are randomised
  and every test has exact size alpha; as lambda grows, X drives both the group
  assignment and the loop count, so ``L(D|A=1) != L(D|A=0)`` while the causal
  nulls hold *exactly*: ``psi_d = 0`` (groups conditionally identical given X)
  and ``delta_dist = 0``.  Reported: rejection rate per (test, lambda): the
  type-I error of the six competitors climbing away from alpha (Theorem 2.1),
  and the DR prototype (task 2.4) holding its level.

Part B -- masking (task 2.2), Figure 1b:
  DGP: ``masking_stratum_sample``: ``L(D|A=1) = L(D|A=0)`` exactly (verified
  algebra and in tests), while ``psi_d != 0`` (Theorem 2.2).  The six
  competitors sit at alpha (their null is exactly true); the DR prototype has
  power.  Diagnostics for the writeup: the per-cloud max-H1-persistence
  difference between arms (centered at 0) and the KS test on pooled
  persistences (uniform under the null).

Modes
-----
* ``--mode shard --shard-idx i --reps-per-shard R``: run replications
  ``[i*R, (i+1)*R)`` of *both* parts in-process (no joblib) and write
  ``results/shards/phase2_shard{i}.json``.  The Colab fleet (see
  ``experiments/colab/``) runs 100 shards x 10 reps = 1000 reps per part,
  five shards per notebook, downloading each shard as it lands.
* ``--mode local --shards 0-3 --workers W``: the same shards, with joblib over
  W workers and one checkpoint file per shard.  Replication indices are keyed
  off the shard, so a shard run here and the same shard run on Colab are
  interchangeable; ``--skip-existing`` resumes an interrupted run.
* ``--mode aggregate``: merge every ``results/shards/phase2_shard*.json``
  (deduplicated by replication index), compute rejection rates, draw
  ``results/phase2_figure1.png`` and the Figure-1 data JSON, and print the
  Phase 2 GATE summary.

Design notes (Section 5 of RESEARCH_PLAN_P1_TwoSample.md):
  * clouds depend on X only (``group_effect=0``), so one set of clouds and
    diagrams serves every lambda: a common-random-numbers design whose group
    splits differ only through the labels.  The DR silhouette triplet is also
    shared.
  * every label-independent structure is built once per replication and read
    back per split (``_precompute``): RT's pairwise bottleneck matrix, MMD's
    diagram Gram matrix and Han's four per-bandwidth kernel matrices.  Those
    three are ~85% of a replication's cost and none of them depends on the
    labels, so rebuilding them for each of the six lambdas buys nothing.
    Group membership enters only as a boolean mask over the pooled rows.
  * near-diagonal diagram points (persistence < ``EPS_RT``) are dropped before
    the RT bottleneck matrix and the MMD/Han kernels.  By stability of the
    bottleneck distance and the boundedness of the Gaussian kernel every entry
    changes by at most ``2 * EPS_RT``, the permutation null remains exactly
    valid for any statistic, and the sweep's signals (loop persistences >=
    ~0.65, loop-count gaps >= ~0.3) sit an order of magnitude above the filter
    error.  The published wrappers are untouched; the filter lives only here
    (see WP2, Section 4).
  * replications are embarrassingly parallel and each shard is a checkpoint,
    so an interrupted run loses at most the shard in flight.  ``--workers``
    caps the pool (default 16) to leave headroom for concurrent work on this
    machine; the Colab notebooks size it from the VM's CPU count.
"""

from __future__ import annotations

import argparse
import json
import os
import time

import numpy as np
from joblib import Parallel, delayed

from tda2s.adapters.dr_test import prototype_dr_from_phi
from tda2s.benchmarks import (han_kernels, mmd_gram, run_competitor,
                              test_han_from_kernels, test_mmd_from_gram,
                              test_rt_from_matrix)
from tda2s.dgp import CloudSampleDGP, masking_stratum_sample, to_silhouette_sample
from tda2s.ph import compute_diagrams

# ----------------------------------------------------------------- config ----
N_PER_GROUP = 200
M = 120
PROP_SCALE = 1.5                       # propensity logit scale at lambda = 1
BETA = np.array([-0.5, -0.1, 0.6])     # tcda_uq default coefficients (d_x = 3)
LAMBDAS = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
D_X = 3
N_PERM = 200
ALPHA = 0.05
INTERVAL = (0.0, 2.0)
RESOLUTION = 100
SIL_R = 3.0
N_BASIS = 8
N_FOLDS = 2
N_DRAWS = 2000
BASE_SEED = 2100
TARGET_REPS = 1000   # the fleet's replication budget; see WP2 section 6
EPS_RT = 0.1     # near-diagonal filter for the RT matrix / MMD / Han kernels
RT_APPROX = 0.01  # gudhi additive tolerance for the bottleneck matrix
COMPETITORS = ["rt", "mmd", "han", "strand", "moon_lazar", "frechet_anova"]
#: competitors with no label-independent precompute, re-run per group split
PER_SPLIT = ["strand", "moon_lazar", "frechet_anova"]
_HERE = os.path.dirname(os.path.abspath(__file__))
RESULTS = os.path.join(_HERE, "..", "results")
SHARDS = os.path.join(RESULTS, "shards")
FIG_PNG = os.path.join(RESULTS, "phase2_figure1.png")
FIG_JSON = os.path.join(RESULTS, "phase2_figure1.json")


def _seed(*parts):
    """Deterministic per-(part, rep, lambda, test) RNG seed."""
    s = BASE_SEED
    for p in parts:
        if isinstance(p, str):
            h = 0
            for ch in p.encode():
                h = (h * 31 + ch) % (2 ** 31)
            p = h
        s = (s * 7919 + int(p)) % (2 ** 31)
    return s


def _diagrams_of(clouds):
    return [compute_diagrams(c, homology_dims=(0, 1)) for c in clouds]


def _rt_matrix(diags):
    """Robinson-Turner pairwise joint-loss matrix, shared across all splits.

    Two implementation notes, both of which leave the permutation null exactly
    valid because the matrix is fixed before any label is drawn:

    * diagram points with persistence < ``EPS_RT`` are dropped (module
      docstring), and
    * the bottleneck calls ask gudhi for an additive ``RT_APPROX``
      approximation instead of the exact CGAL path, which is ~4x faster here.
      Every entry moves by at most ``2 * RT_APPROX = 0.02`` against a signal
      (loop persistences >= ~0.65) an order of magnitude larger.

    Empty (post-filter) diagrams are *not* skipped: ``d_B(varnothing, D)`` is
    half the largest persistence of ``D``, and in this sweep the group contrast
    is precisely a difference in loop *count*, so a diagram that filters down
    to nothing is the informative case, not a missing one.
    """
    n = len(diags)
    P = np.zeros((n, n))
    gd = __import__("gudhi", fromlist=["bottleneck_distance"])
    filt = [[d[d[:, 1] - d[:, 0] >= EPS_RT] for d in diag] for diag in diags]
    for dim in range(2):
        for i in range(n):
            Di = filt[i][dim]
            for j in range(i + 1, n):
                v = float(gd.bottleneck_distance(Di, filt[j][dim], RT_APPROX))
                P[i, j] = P[j, i] = P[i, j] + v
    return P


def _precompute(diags):
    """Everything about one replication that does not depend on the labels.

    RT's pairwise bottleneck matrix, MMD's diagram Gram matrix and Han's four
    per-bandwidth kernel matrices are functions of the *pooled* diagrams alone,
    so a replication that compares six propensity strengths over one sample
    builds them once here and reads them back per split. Rebuilding them per
    lambda is ~85% of the replication's cost and buys nothing.
    """
    return {"rt": _rt_matrix(diags),
            "mmd": mmd_gram(diags, epsilon=EPS_RT)[0],
            "han": han_kernels(diags, epsilon=EPS_RT)}


def _competitor_pvalues(diags, pre, A, seed):
    """All six competitor p-values for one group split of a replication.

    Args:
        diags: the pooled diagrams, in the replication's own order.
        pre: the label-independent structures from :func:`_precompute`.
        A: ``(N,)`` treatment labels in that same order.
        seed: base RNG seed for this split.

    The three precomputed tests read their matrices back under the mask
    ``A == 0``; the remaining three have no label-independent structure worth
    caching and are re-run on the split diagram lists.
    """
    mask = np.asarray(A).astype(bool)
    g0 = ~mask                       # True = group 0, in the pooled row order
    d0 = [d for d, m in zip(diags, mask) if not m]
    d1 = [d for d, m in zip(diags, mask) if m]

    out = {name: run_competitor(name, d0, d1, n_perm=N_PERM, seed=seed,
                                epsilon=EPS_RT)
           for name in PER_SPLIT}
    out["rt"] = float(test_rt_from_matrix(pre["rt"], g0, n_perm=N_PERM,
                                          statistic="within", seed=seed))
    out["mmd"] = float(test_mmd_from_gram(pre["mmd"], g0, n_perm=N_PERM,
                                          seed=seed))
    out["han"] = float(test_han_from_kernels(pre["han"], g0, n_perm=N_PERM,
                                             seed=seed))
    return out


# ---------------------------------------------------------------- part A -----
def expit_lambda(dgp, X, lam):
    return 1.0 / (1.0 + np.exp(-PROP_SCALE * lam * (np.asarray(X) @ dgp.beta)))


def rep_false_positive(rep):
    """One replication of the Part A sweep; returns per-lambda p-values.

    Every RNG draw is keyed off ``rep`` (and, for the labels, off lambda), so a
    replication is reproducible in isolation: shard *i* run on Colab and the
    same replication index run here give the same numbers.
    """
    seed = _seed("A", rep)
    dgp = CloudSampleDGP(n_per_group=N_PER_GROUP, m=M, d_x=D_X, beta=BETA,
                         prop_scale=PROP_SCALE, group_effect=0, seed=seed)
    sample = dgp.sample(rng=seed)
    clouds, X = sample.clouds, sample.X
    n = len(clouds)

    diags = _diagrams_of(clouds)
    pre = _precompute(diags)
    phi, _, _ = to_silhouette_sample(clouds, X, np.zeros(n), filtration="alpha",
                                     homology_dims=(0, 1), interval=INTERVAL,
                                     r=SIL_R, resolution=RESOLUTION)
    tseq = np.linspace(INTERVAL[0], INTERVAL[1], RESOLUTION)

    out = {}
    for lam in LAMBDAS:
        key = f"lam{lam:g}"
        pi = expit_lambda(dgp, X, lam)
        labels_rng = np.random.default_rng(_seed("A", rep, key, "labels"))
        A = labels_rng.binomial(1, pi).astype(int)
        pvals = _competitor_pvalues(diags, pre, A, _seed("A", rep, key, "t"))
        pvals["dr"] = prototype_dr_from_phi(phi, A, X, tseq, n_basis=N_BASIS,
                                            n_folds=N_FOLDS, n_draws=N_DRAWS,
                                            seed=_seed("A", rep, key, "dr"))
        out[key] = pvals
    return {"rep": rep, "pvals": out}


# ---------------------------------------------------------------- part B -----
def rep_masking(rep):
    """One replication of the Part B masking experiment."""
    seed = _seed("B", rep)
    sample = masking_stratum_sample(N_PER_GROUP, seed=seed)
    clouds, X, A = sample.clouds, sample.X, sample.A

    diags = _diagrams_of(clouds)
    pre = _precompute(diags)
    phi, _, _ = to_silhouette_sample(clouds, X, A, filtration="alpha",
                                     homology_dims=(0, 1), interval=INTERVAL,
                                     r=SIL_R, resolution=RESOLUTION)
    tseq = np.linspace(INTERVAL[0], INTERVAL[1], RESOLUTION)

    pvals = _competitor_pvalues(diags, pre, A, _seed("B", rep, "t"))
    pvals["dr"] = prototype_dr_from_phi(phi, A, X, tseq, n_basis=N_BASIS,
                                        n_folds=N_FOLDS, n_draws=N_DRAWS,
                                        seed=_seed("B", rep, "dr"))

    pers1 = np.concatenate([d[1][:, 1] - d[1][:, 0] for d, m in zip(diags, A)
                            if m and d[1].size])
    pers0 = np.concatenate([d[1][:, 1] - d[1][:, 0] for d, m in zip(diags, A)
                            if not m and d[1].size])
    mean_diff = float(pers1.mean() - pers0.mean()) if pers1.size and pers0.size else None
    ks_p = None
    if pers1.size and pers0.size:
        from scipy import stats
        ks_p = float(stats.ks_2samp(pers1, pers0).pvalue)
    return {"rep": rep, "pvals": pvals, "mean_pers_diff": mean_diff, "ks_p": ks_p}


# ---------------------------------------------------------------- shards -----
def run_shard(shard_idx, reps_per_shard, workers=1):
    """Run replications ``[shard_idx*R, (shard_idx+1)*R)`` of both parts.

    The file is named after the *replication range*, never after where it ran,
    so a shard computed locally and the same shard computed on Colab are
    interchangeable and the aggregate deduplicates them cleanly.

    Returns the path of the written ``results/shards/phase2_shard<i>.json``.
    """
    start = int(shard_idx) * int(reps_per_shard)
    reps = list(range(start, start + int(reps_per_shard)))
    t0 = time.time()
    if workers > 1:
        part_a = Parallel(n_jobs=workers, verbose=1)(
            delayed(rep_false_positive)(r) for r in reps)
        part_b = Parallel(n_jobs=workers, verbose=1)(
            delayed(rep_masking)(r) for r in reps)
    else:
        part_a = [rep_false_positive(r) for r in reps]
        part_b = [rep_masking(r) for r in reps]
    name = f"shard{shard_idx}"
    path = os.path.join(SHARDS, f"phase2_{name}.json")
    os.makedirs(SHARDS, exist_ok=True)
    payload = {
        "name": name,
        "config": {"n_per_group": N_PER_GROUP, "m": M, "prop_scale": PROP_SCALE,
                   "beta": list(BETA), "lambdas": list(LAMBDAS),
                   "n_perm": N_PERM, "n_draws": N_DRAWS, "epsilon": EPS_RT,
                   "reps_per_shard": int(reps_per_shard),
                   "start": start, "reps": reps},
        "part_a": part_a, "part_b": part_b,
    }
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(payload, fh)
    os.replace(tmp, path)
    dt = time.time() - t0
    print(f"[phase2] shard {name}: {len(reps)} reps x 2 parts in {dt:.0f}s "
          f"-> {path}  (per-rep avg {dt / (2 * len(reps)):.1f}s)")
    return path


#: keys of ``payload["config"]`` that must agree across shards to be poolable.
_POOLABLE = ("n_per_group", "m", "prop_scale", "beta", "lambdas", "n_perm",
             "n_draws", "epsilon")


def _load_shards():
    """Merge every ``phase2_shard<i>.json``, deduplicated by replication index.

    Only files named for a replication range are read: anything else in the
    directory (a scratch run, a smoke test) is reported and skipped rather than
    silently pooled, since a stray file whose replication indices overlap a
    real shard would win the deduplication and quietly replace it.

    Shards whose sampling configuration disagrees with the first are refused
    outright: pooling replications run under different ``n_per_group`` or
    ``lambdas`` would produce a rejection rate that estimates nothing.
    """
    all_files = sorted(f for f in os.listdir(SHARDS) if f.endswith(".json"))
    files = [f for f in all_files if f.startswith("phase2_shard")]
    skipped = [f for f in all_files if f not in files]
    if skipped:
        print(f"[phase2] ignoring {len(skipped)} non-shard file(s) in {SHARDS}: "
              f"{', '.join(skipped)}")
    if not files:
        raise SystemExit(f"no phase2_shard*.json under {SHARDS}; "
                         f"run --mode shard/local first")
    config, part_a, part_b = None, {}, {}
    for fn in files:
        with open(os.path.join(SHARDS, fn)) as fh:
            data = json.load(fh)
        cfg = data["config"]
        if config is None:
            config = cfg
        else:
            bad = [k for k in _POOLABLE if cfg.get(k) != config.get(k)]
            if bad:
                raise SystemExit(
                    f"{fn} disagrees with the other shards on {bad}; "
                    f"re-run it against the current config before aggregating")
        for r in data["part_a"]:
            part_a.setdefault(r["rep"], r)
        for r in data["part_b"]:
            part_b.setdefault(r["rep"], r)
    print(f"[phase2] merged {len(files)} shard file(s): "
          f"{len(part_a)} part-A and {len(part_b)} part-B replications")
    _report_coverage(part_a, part_b)
    return config, part_a, part_b


def _report_coverage(part_a, part_b, target=TARGET_REPS):
    """Report which replication indices are missing from the merged fleet.

    With a hundred shard files arriving from a notebook fleet, a dropped
    download is easy to miss and silently costs power rather than raising
    anything, so the gap is named explicitly.
    """
    for label, part in (("part A", part_a), ("part B", part_b)):
        have = set(part)
        if not have:
            continue
        missing = sorted(set(range(target)) - have)
        extra = sorted(r for r in have if r >= target)
        if not missing and not extra:
            print(f"[phase2] {label}: complete, replications 0-{target - 1}")
            continue
        # collapse the missing indices into the shard ranges they came from
        shards = sorted({r // 10 for r in missing})
        head = ", ".join(str(s) for s in shards[:12])
        tail = "" if len(shards) <= 12 else f", ... (+{len(shards) - 12} more)"
        print(f"[phase2] {label}: {len(have)}/{target} replications; "
              f"{len(shards)} shard(s) missing at 10 reps/shard: {head}{tail}")
        if extra:
            print(f"[phase2] {label}: {len(extra)} replication(s) beyond "
                  f"index {target - 1} were also merged")


# ---------------------------------------------------------------- figure -----
def _rates_a(part_a, keys):
    tests = list(COMPETITORS) + ["dr"]
    rates = {t: {k: 0.0 for k in keys} for t in tests}
    for rep in part_a.values():
        for k in keys:
            for t in tests:
                rates[t][k] += (rep["pvals"][k][t] <= ALPHA)
    n = max(1, len(part_a))
    for t in tests:
        for k in keys:
            rates[t][k] /= n
    return rates


def _rates_b(part_b):
    tests = list(COMPETITORS) + ["dr"]
    rates = {t: 0.0 for t in tests}
    for rep in part_b.values():
        for t in tests:
            rates[t] += (rep["pvals"][t] <= ALPHA)
    n = max(1, len(part_b))
    for t in tests:
        rates[t] /= n
    return rates


def make_figure(sweep_rates, masking_rates, n_reps):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5.4))
    tests = list(COMPETITORS) + ["dr"]
    labels = {"rt": "Robinson-Turner", "mmd": "MMD", "han": "Han et al.",
              "strand": "STRAND", "moon_lazar": "Moon-Lazar",
              "frechet_anova": "Frechet ANOVA", "dr": "DR prototype"}
    lam_keys = [f"lam{lam:g}" for lam in LAMBDAS]
    xs = np.arange(len(lam_keys))
    for t in tests:
        ax1.plot(xs, [sweep_rates[t][k] for k in lam_keys], marker="o",
                 ms=4, lw=1.4, label=labels[t])
    ax1.axhline(ALPHA, color="k", ls="--", lw=0.9)
    ax1.text(0.01, 0.045, r"$\alpha = 0.05$", fontsize=8)
    # a rejection rate cannot be negative, so clip the band rather than let it
    # run below the axis at small replication counts
    se = np.sqrt(ALPHA * (1 - ALPHA) / max(1, n_reps))
    ax1.fill_between(xs, max(0.0, ALPHA - 3 * se), ALPHA + 3 * se, color="k",
                     alpha=0.12,
                     label=f"$\\alpha \\pm 3\\,\\mathrm{{SE}}$ ({n_reps} reps)")
    ax1.set_xticks(xs, [f"{lam:g}" for lam in LAMBDAS])
    ax1.set_ylim(-0.02, 1.02)
    ax1.set_xlabel("imbalance $\\lambda$ (propensity logit scale)")
    ax1.set_ylabel("type-I error rate at $\\alpha = 0.05$")
    ax1.set_title("(a) false positives under covariate shift ($\\psi_d \\equiv 0$)")
    ax1.legend(fontsize=7.5, loc="upper left")
    ax1.grid(alpha=0.25)

    y = [masking_rates[t] for t in tests]
    bars = ax2.bar(np.arange(len(tests)), y, color=["#4477AA"] * 6 + ["#CC6677"])
    ax2.axhline(ALPHA, color="k", ls="--", lw=0.9)
    ax2.set_xticks(np.arange(len(tests)), [labels[t] for t in tests], rotation=28, fontsize=8)
    ax2.set_ylabel("rejection rate at $\\alpha = 0.05$")
    ax2.set_ylim(0, 1.12)   # headroom for the bar value labels
    ax2.set_title("(b) Simpson masking ($L(D|A{=}1)=L(D|A{=}0)$, $\\psi_d \\neq 0$)")
    ax2.grid(axis="y", alpha=0.25)
    for b, v in zip(bars, y):
        ax2.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}",
                 ha="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_PNG, dpi=200)
    return fig


def aggregate(print_gate=True):
    """Merge shards, write Figure 1 + data JSON, print the gate report."""
    config, part_a, part_b = _load_shards()
    n_a, n_b = len(part_a), len(part_b)
    sweep_rates = _rates_a(part_a, [f"lam{lam:g}" for lam in LAMBDAS])
    masking_rates = _rates_b(part_b)

    make_figure(sweep_rates, masking_rates, n_a)
    fig_data = {
        "n_reps": {"sweep": n_a, "masking": n_b},
        "alpha": ALPHA,
        "config": config,
        "sweep_rates": sweep_rates,
        "masking_rates": masking_rates,
    }
    with open(FIG_JSON, "w") as fh:
        json.dump(fig_data, fh, indent=1)
    print(f"[phase2] figure written to {FIG_PNG}, data to {FIG_JSON}")
    if print_gate:
        print_gate_report(sweep_rates, masking_rates, part_b, n_a)
    return fig_data


def print_gate_report(sweep_rates, masking_rates, part_b, n_reps):
    """The task 2.5 verdict, as stated in RESEARCH_PLAN_P1_TwoSample.md.

    The plan's criterion is a *disjunction* over the two failure modes, both
    conditioned on the DR prototype keeping its level:

        (2.1 false positives at the strongest imbalance  OR  2.2 masking)
        AND  the DR prototype holds size across the sweep.

    The labels below are the plan's *task* numbers, not its contribution
    labels: both failure modes are evidence for contribution C1, which is why
    either one of them passes the gate. 2.1 is read at ``lambda = 1``
    specifically, not as a maximum over the sweep:
    the claim being certified is that the competitors fail *at the strongest
    imbalance*, and a maximum over six settings would also fire on a single
    lucky interior point.

    One refinement of the plan's binary verdict. The plan's FAIL branch says
    "drop C1, rewrite the abstract around C2+C3", which is the right response
    when the *competitors* turn out to be fine under imbalance. It is not the
    right response when the competitor evidence is overwhelming and the only
    unmet condition is the size of the deliberately uncalibrated Phase 2.4
    prototype (task 2.4's own words: "rough, uncalibrated ... polish in Phase
    3"). That combination says the prototype needs Phase 3's calibration (3.2:
    proper multiplier bootstrap and the stratified-permutation variant), not
    that C1 is false. It is reported as INCONCLUSIVE and left as the caller's
    call, rather than silently counted as either outcome.
    """
    lam_keys = [f"lam{lam:g}" for lam in LAMBDAS]
    worst_at_1 = max(sweep_rates[t]["lam1"] for t in COMPETITORS)
    worst_over_sweep = max(max(sweep_rates[t].values()) for t in COMPETITORS)
    dr_size = [sweep_rates["dr"][k] for k in lam_keys]
    max_comp_mask = max(masking_rates[t] for t in COMPETITORS)
    dr_mask = masking_rates["dr"]

    fp_ok = worst_at_1 >= 0.20
    mask_ok = max_comp_mask <= 0.10 and dr_mask >= 0.70
    size_ok = all(0.03 <= v <= 0.08 for v in dr_size)
    se = np.sqrt(ALPHA * (1 - ALPHA) / max(1, n_reps))

    provisional = n_reps < TARGET_REPS
    banner = ("PROVISIONAL (partial fleet)" if provisional else "")
    print(f"\n===== Phase 2 GATE summary {banner} =====")
    print(f"  replications: {n_reps}  (MC se at alpha: {se:.4f})")
    if provisional:
        print(f"  ** {n_reps} of {TARGET_REPS} replications. The size band "
              f"[0.03, 0.08] is +-2 MC se at 200 replications, so a verdict "
              f"read off this subset is not the gate. **")
    print(f"  2.1 false positives: worst competitor at lambda=1 = {worst_at_1:.3f} "
          f"(need >= 0.20) -> {'MET' if fp_ok else 'not met'}")
    print(f"                       (worst anywhere in the sweep: {worst_over_sweep:.3f})")
    print(f"  2.2 masking        : competitor max = {max_comp_mask:.3f} "
          f"(need <= 0.10), DR = {dr_mask:.3f} (need >= 0.70) "
          f"-> {'MET' if mask_ok else 'not met'}")
    print(f"  2.4 DR size        : {min(dr_size):.3f}-{max(dr_size):.3f} across lambda "
          f"(need every lambda in [0.03, 0.08]) -> {'MET' if size_ok else 'not met'}")
    if not size_ok:
        bad = [f"{k}={v:.3f}" for k, v in zip(lam_keys, dr_size)
               if not 0.03 <= v <= 0.08]
        print(f"                       outside the band: {', '.join(bad)}")
    diffs = np.array([r["mean_pers_diff"] for r in part_b.values()
                      if r["mean_pers_diff"] is not None])
    if diffs.size:
        print(f"  diagnostic         : E[H1 pers | A=1] - E[H1 pers | A=0] = "
              f"{diffs.mean():+.4f} (sd {diffs.std():.4f}, want ~0)")
    evidence = fp_ok or mask_ok
    if evidence and size_ok:
        verdict, action = "PASS", ("C1 is the spine; proceed to Phases 3-7 "
                                   "as written")
    elif evidence:
        verdict, action = "INCONCLUSIVE", (
            "the competitor failure is established, but the uncalibrated "
            "prototype does not hold its level; this is Phase 3.2's job, not "
            "a refutation of C1. Calibrate, then re-fire")
    else:
        verdict, action = "FAIL", ("drop C1, rewrite around C2+C3 "
                                   "(see plan section 3, Phase 2)")
    print(f"  GATE{' (provisional)' if provisional else ''}: {verdict} -> {action}")
    return verdict


# ----------------------------------------------------------------- main ------
def _parse_shards(spec):
    """``"3"`` / ``"0-7"`` / ``"0,2,5-7"`` -> a list of shard indices."""
    out = []
    for part in str(spec).split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            lo, hi = part.split("-", 1)
            out.extend(range(int(lo), int(hi) + 1))
        else:
            out.append(int(part))
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--mode", choices=["shard", "local", "aggregate"], required=True)
    ap.add_argument("--shard-idx", type=int, default=0,
                    help="shard index for --mode shard (one notebook = one shard)")
    ap.add_argument("--shards", default=None,
                    help="shard indices for --mode local, e.g. '0-7' or '0,3,5'; "
                         "defaults to --shard-idx")
    ap.add_argument("--reps-per-shard", type=int, default=25)
    ap.add_argument("--workers", type=int, default=16)
    ap.add_argument("--skip-existing", action="store_true",
                    help="--mode local: leave already-written shard files alone")
    args = ap.parse_args()

    if args.mode == "shard":
        run_shard(args.shard_idx, args.reps_per_shard, workers=1)
    elif args.mode == "local":
        idxs = _parse_shards(args.shards) if args.shards else [args.shard_idx]
        for i in idxs:
            path = os.path.join(SHARDS, f"phase2_shard{i}.json")
            if args.skip_existing and os.path.exists(path):
                print(f"[phase2] shard {i} already at {path}, skipping")
                continue
            run_shard(i, args.reps_per_shard, workers=args.workers)
    else:
        aggregate()


if __name__ == "__main__":
    main()


In [ ]:
import os, sys

sys.path.insert(0, "/content")
os.environ["PYTHONPATH"] = "/content"   # joblib workers inherit this

import gudhi, numpy, tcda_uq                      # noqa: F401
from experiments.phase2_imbalance_sweep import LAMBDAS, N_PER_GROUP, run_shard

print("imports OK |", N_PER_GROUP, "units per arm |", len(LAMBDAS), "lambdas |",
      os.cpu_count(), "CPUs")


In [ ]:
SHARD_IDXS = [85, 86, 87, 88, 89]
REPS_PER_SHARD = 10

import os, time

# one worker per CPU, capped: the per-replication peak is ~1 GB and Colab's
# free VM has ~12 GB, so the cap is about leaving the VM responsive.
WORKERS = max(1, min(os.cpu_count() or 1, 4))
print(f"running shards {SHARD_IDXS} on {WORKERS} worker(s)")

t0 = time.time()
for shard in SHARD_IDXS:
    path = run_shard(shard, REPS_PER_SHARD, workers=WORKERS)
    try:
        from google.colab import files
        files.download(path)
        print("Downloaded:", path)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)
    print(f"--- {time.time() - t0:.0f}s elapsed, "
          f"{SHARD_IDXS.index(shard) + 1}/{len(SHARD_IDXS)} shards done ---")

print("all shards done in", round(time.time() - t0), "s")
print("files also kept in /content/results/shards/ if a download was missed")
